In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! pip install mne

In [ ]:
#%matplotlib inline
#%tensorflow_version 1.x
import tensorflow as tf
import pandas as pd
import numpy as np
from tqdm import notebook
from sklearn.decomposition import FastICA
import numba as nb
import seaborn as sns
import mne
import scipy
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import mne
import scipy
from joblib import parallel, delayed

import numpy as np
import pandas as pd
from mne.io import RawArray
from mne import EpochsArray
#import mne.preprocessing.compute_current_source_density
#from mne.channels import read_montage
from mne.epochs import concatenate_epochs
from mne import create_info, find_events, Epochs
#from mne.viz.topomap import _prepare_topo_plot, plot_topomap
from mne.decoding import CSP

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from glob import glob

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from scipy.signal import welch
from mne import pick_types
from subprocess import check_output

from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs,corrmap)
from mne.time_frequency.tfr import morlet
from mne.viz import plot_filter, plot_ideal_filter
#import eeglabio
#from tensorflow.keras.preprocessing.image import ImageDataGenerator
#from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import numpy as np
import seaborn as sns
#get_ipython().system('pip install --quiet shap')
#import shap
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
#from keras.applications.vgg16 import VGG16,preprocess_input
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score,cohen_kappa_score,roc_auc_score,confusion_matrix,classification_report


In [ ]:
## Path to access files (change the path)
## ADHD--MIPDB dataset
path = '/content/drive/My Drive/XXX/Dataset/Dementia/'
path = '/content/drive/My Drive/XXX/Dataset/Healthy/'
path_save= '/content/drive/My Drive/XXX/Temp_Gated_Connectivity/'


In [ ]:
## Step-1: Read existing .set file (EEGLAB)
import mne

# Path to your EEGLAB files

filename = "sub-013_task-eyesclosed_eeg.set"
fn = path + filename   # keep the paired .fdt in the SAME folder if it exists
#fn = path_set + "S1_HBN_surroundSupp_run-1_eeg.set"
subject_id = filename.split("_")[0]   # --> 'sub-067'
# Read as Raw (continuous)
raw = mne.io.read_raw_eeglab(fn, preload=True)

# Events (EEGLAB events become MNE Annotations)
events, event_id = mne.events_from_annotations(raw)

print(raw)
print("Event IDs:", event_id)


In [ ]:
## Step 2: preprocessing and epoch creation

from pathlib import Path
import numpy as np
import mne

# ----------------------------
# 1) Load EEGLAB .set
# ----------------------------
raw_eeg = mne.io.read_raw_eeglab(fn, preload=True)

# Optional: Keep only EEG channels (if file contains other channels)
raw_eeg.pick_types(eeg=True, eog=False, ecg=False, emg=False, stim=False, exclude=[])

# Optional: Set montage if channel locations are missing or incomplete
# raw_eeg.set_montage("standard_1020", match_case=False)

# ----------------------------
# 2) Basic filtering (edit as needed)
# ----------------------------
# Choose your line noise frequency: 50 (most countries) or 60 (US)
line_freq = 50

# Notch at line noise and harmonics up to Nyquist
sfreq = raw_eeg.info["sfreq"]
nyquist = sfreq / 2.0
notch_freqs = np.arange(line_freq, nyquist, line_freq)
if len(notch_freqs) > 0:
    raw_eeg.notch_filter(freqs=notch_freqs, picks="eeg")

# Bandpass filter (typical ranges: 0.5–40 Hz or 1–45 Hz for resting EEG)
raw_eeg.filter(l_freq=0.5, h_freq=40.0, picks="eeg")

# ----------------------------
# 3) Re-reference (common choice: average reference)
# ----------------------------
raw_eeg.set_eeg_reference(ref_channels="average", projection=False)

# ----------------------------
# 4) Apply ICA to remove artifacts
# ----------------------------
ica = mne.preprocessing.ICA(n_components=raw_eeg.info['nchan'], random_state=97, max_iter='auto')
ica.fit(raw_eeg)
ica.apply(raw_eeg)

# ----------------------------
# 5) Epoching: 4 s windows with 50% overlap
# ----------------------------
epoch_len_sec = 4.0
overlap_ratio = 0.50
overlap_sec = epoch_len_sec * overlap_ratio  # 2.0 sec overlap

epochs = mne.make_fixed_length_epochs(
    raw_eeg,
    duration=epoch_len_sec,
    overlap=overlap_sec,
    preload=True,
    reject_by_annotation=True  # excludes segments marked "bad"
)

epochs.apply_baseline((None, None))

# ----------------------------
# 6) Output: epochs object + numpy array
# ----------------------------
X = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
print("Epochs:", epochs)
print("X shape:", X.shape)
print("Sampling rate:", epochs.info["sfreq"])



In [ ]:
##Step2: Detect ERP peaks with windows

# Step 4.2 — Detect ERP peaks with windows (adaptive number of peaks)
import numpy as np
from scipy.signal import find_peaks
import mne
from typing import Optional, Tuple, List, Dict, Any

def _get_erp_trace_for_peak_detection(
    evoked: mne.Evoked,
    method: str = "gfp",
    channel: Optional[str] = None
) -> np.ndarray:
    """
    Returns a 1D trace over time for peak detection.

    method:
      - "channel": use a single channel (requires `channel`)
      - "mean": mean across EEG channels
      - "gfp": global field power (std across channels) (nonnegative)  <-- recommended for "brain activation"
    """
    if method == "channel":
        if channel is None:
            raise ValueError("method='channel' requires channel name, e.g., channel='Cz'")
        ev = evoked.copy().pick([channel])
        return ev.data[0]

    if method == "mean":
        ev = evoked.copy().pick_types(eeg=True)
        return ev.data.mean(axis=0)

    if method == "gfp":
        ev = evoked.copy().pick_types(eeg=True)
        return ev.data.std(axis=0)  # GFP >= 0

    raise ValueError(f"Unknown method: {method}")


def _robust_scale_mad(x: np.ndarray, eps: float = 1e-12) -> float:
    """
    Robust scale estimator using MAD (median absolute deviation).
    scale ≈ 1.4826 * MAD for Gaussian-like data.
    """
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    scale = 1.4826 * mad
    # fallback if flat
    if not np.isfinite(scale) or scale < eps:
        scale = float(np.std(x)) + eps
    return float(scale)


def detect_erp_peaks_by_activation(
    epochs_train: mne.Epochs,
    search_range_sec: Optional[Tuple[float, float]] = (0.0, 0.8),
    min_distance_ms: float = 80.0,
    trace_method: str = "gfp",            # <-- "brain activation" default
    channel: Optional[str] = None,
    # adaptive thresholding controls:
    prominence_z: float = 3.0,            # higher -> fewer peaks (stricter)
    height_z: Optional[float] = None,     # optionally also enforce a height threshold
    max_peaks: Optional[int] = None       # optional cap if too many peaks are found
):
    """
    Adaptive ERP peak detection:
      - No fixed n_peaks.
      - Detect peaks using GFP (default) or mean/channel trace.
      - Uses robust MAD-based prominence threshold to keep only meaningful peaks.

    Returns:
      evoked, peaks_list
    """
    sfreq = float(epochs_train.info["sfreq"])
    evoked = epochs_train.average()

    trace = _get_erp_trace_for_peak_detection(evoked, method=trace_method, channel=channel)

    # For mean/channel traces, use abs to catch both pos/neg deflections.
    # For GFP, trace is already nonnegative (activation magnitude).
    if trace_method in ("mean", "channel"):
        detect_sig_full = np.abs(trace)
    else:
        detect_sig_full = trace

    times = evoked.times

    # restrict search region
    if search_range_sec is not None:
        tmin, tmax = search_range_sec
        idx_min = int(np.searchsorted(times, tmin))
        idx_max = int(np.searchsorted(times, tmax))
    else:
        idx_min, idx_max = 0, len(times)

    detect_sig = detect_sig_full[idx_min:idx_max]
    if detect_sig.size < 5:
        raise ValueError("Search range too small / not enough samples for peak detection.")

    # Adaptive thresholds from robust statistics of the detection signal
    scale = _robust_scale_mad(detect_sig)
    prom_thresh = float(prominence_z * scale)

    # Optional height threshold
    height_thresh = None
    if height_z is not None:
        height_thresh = float(np.median(detect_sig) + height_z * scale)

    # Minimum distance in samples (avoid peaks too close)
    min_distance_samp = max(1, int(round((min_distance_ms / 1000.0) * sfreq)))

    peaks_local, props = find_peaks(
        detect_sig,
        distance=min_distance_samp,
        prominence=prom_thresh,
        height=height_thresh
    )

    if len(peaks_local) == 0:
        raise RuntimeError(
            "No peaks found with current thresholds.\n"
            "Try: lower prominence_z (e.g., 2.0), set search_range_sec=None, "
            "or switch trace_method to 'gfp' if not already."
        )

    # If too many peaks, optionally keep the strongest by prominence
    if max_peaks is not None and len(peaks_local) > max_peaks:
        prominences = props.get("prominences", np.ones(len(peaks_local)))
        order = np.argsort(prominences)[::-1][:max_peaks]
        peaks_local = np.sort(peaks_local[order])

    # Convert local -> global indices
    peaks_global = peaks_local + idx_min

    # Build peak dict list (sorted in time order already)
    peaks: List[Dict[str, Any]] = []
    for k, gi in enumerate(peaks_global, start=1):
        amp_signed = float(trace[gi])            # signed amplitude in original trace
        amp_abs = float(abs(amp_signed))
        peaks.append({
            "k": k,
            "sample_index": int(gi),
            "time_sec": float(times[gi]),
            "amp": amp_signed,
            "abs_amp": amp_abs,
            "trace_method": trace_method,
        })

    return evoked, peaks


def compute_peak_windows_from_sfreq(
    peaks: List[Dict[str, Any]],
    sfreq: float,
    half_width_ms: float = 40.0,
    n_times: Optional[int] = None
):
    """
    Compute peak-centered windows [t_k - Δ, t_k + Δ] in sample indices.

    half_width_ms = Δ in milliseconds (half-width)
    At 500 Hz: 40 ms -> 0.040*500 = 20 samples (on EACH side).
    """
    delta_samples = int(round((half_width_ms / 1000.0) * sfreq))  # Δ in samples (each side)

    windows = []
    for pk in peaks:
        c = int(pk["sample_index"])
        s = c - delta_samples
        e = c + delta_samples

        if n_times is not None:
            s = max(0, s)
            e = min(n_times - 1, e)

        windows.append({
            "k": int(pk["k"]),
            "center_samp": c,
            "start_samp": int(s),
            "end_samp": int(e),  # inclusive end (your convention)
            "delta_samples": int(delta_samples),
            "half_width_ms": float(half_width_ms),
            "center_time_sec": float(pk["time_sec"]),
        })

    return windows


# -----------------------------
# Example usage
# -----------------------------
epochs_train = epochs  # in CV: use TRAIN only

# 1) Detect peaks adaptively (no fixed number)
evoked_train, peaks = detect_erp_peaks_by_activation(
    epochs_train=epochs_train,
    search_range_sec=None,   # set None if you want full epoch
    min_distance_ms=80.0,
    trace_method="gfp",            # "brain activation" default
    prominence_z=3.0,              # lower -> more peaks; higher -> fewer peaks
    height_z=None,                 # optionally set (e.g., 1.0) to be stricter
    max_peaks=None                 # optionally set to cap (e.g., 8) if needed
)

print(f"Detected {len(peaks)} peaks:")
for p in peaks:
    print(p)

# 2) Compute windows around each peak
sfreq = float(epochs_train.info["sfreq"])
n_times = len(evoked_train.times)

peak_windows = compute_peak_windows_from_sfreq(
    peaks=peaks,
    sfreq=sfreq,
    half_width_ms=40.0,  # Δ = 40 ms => 20 samples each side at 500 Hz (total ~80 ms window)
    n_times=n_times
)

print("\nPeak windows (samples):")
for w in peak_windows:
    print(w)


In [ ]:
# Step 4.3: Peak-locked connectivity graphs (partial correlation) + Save (subject_id from filename)

import numpy as np
from pathlib import Path
import json

# -----------------------------
# SUBJECT / FILE INFO
# -----------------------------


# If you want a full file path (optional, useful for logging / metadata)
# Make sure `path` is defined, e.g., path = "/content/drive/MyDrive/.../raw/"
#file_path = Path(path) / filename

print("filename:", filename)
print("subject_id:", subject_id)
#print("file_path:", file_path)

# -----------------------------
# Partial correlation adjacency
# -----------------------------
# Try to use LedoitWolf shrinkage if scikit-learn is available (recommended)
try:
    from sklearn.covariance import LedoitWolf
    _HAS_SKLEARN = True
except Exception:
    LedoitWolf = None
    _HAS_SKLEARN = False


def partial_corr_from_window(
    X_ch_time: np.ndarray,
    use_abs: bool = True,
    eps: float = 1e-12,
    ridge: float = 1e-6,
) -> np.ndarray:
    """
    Compute partial correlation adjacency from one window.

    X_ch_time: (n_channels, n_samples_in_window)
    Returns:   (n_channels, n_channels) partial correlation matrix (diag=0, symmetric)
    """
    X = X_ch_time.T  # (m, N)
    X = X - X.mean(axis=0, keepdims=True)

    n_features = X.shape[1]

    if _HAS_SKLEARN:
        lw = LedoitWolf(assume_centered=True)
        lw.fit(X)
        precision = lw.precision_
    else:
        cov = np.cov(X, rowvar=False, bias=False)
        avg_var = float(np.trace(cov) / max(n_features, 1))
        cov = cov + (ridge + 1e-3 * avg_var) * np.eye(n_features)
        precision = np.linalg.pinv(cov)

    d = np.sqrt(np.clip(np.diag(precision), eps, None))
    denom = np.outer(d, d)
    A_pcorr = -precision / np.clip(denom, eps, None)

    np.fill_diagonal(A_pcorr, 0.0)
    A_pcorr = 0.5 * (A_pcorr + A_pcorr.T)

    if use_abs:
        A_pcorr = np.abs(A_pcorr)

    return A_pcorr


def build_peak_locked_partial_corr_graphs(
    epochs,
    peak_windows: list[dict],
    use_abs: bool = True,
):
    """
    epochs.get_data(): (n_epochs, n_channels, n_times)
    returns A_all:     (n_epochs, n_peaks, n_channels, n_channels)
    """
    X = epochs.get_data()
    n_epochs, n_channels, n_times = X.shape
    n_peaks = len(peak_windows)

    A_all = np.zeros((n_epochs, n_peaks, n_channels, n_channels), dtype=np.float32)

    for pk_idx, w in enumerate(peak_windows):
        start = int(w["start_samp"])
        end_inclusive = int(w["end_samp"])
        end = end_inclusive + 1  # inclusive -> exclusive for slicing

        start = max(0, start)
        end = min(n_times, end)

        if end - start < 3:
            raise ValueError(
                f"Window too short for peak k={w.get('k', pk_idx+1)}: "
                f"start={start}, end={end} (length={end-start} samples)."
            )

        for n in range(n_epochs):
            X_win = X[n, :, start:end]
            A_all[n, pk_idx] = partial_corr_from_window(X_win, use_abs=use_abs)

    return A_all


# -----------------------------
# Compute graphs
# -----------------------------
A_pcorr = build_peak_locked_partial_corr_graphs(
    epochs=epochs,
    peak_windows=peak_windows,
    use_abs=True
)

print("A_pcorr shape:", A_pcorr.shape)  # (n_epochs, n_peaks, n_channels, n_channels)
print("Diagonal (first epoch, first peak):", np.diag(A_pcorr[0, 0]))
print("Symmetry check (max |A-A.T|):", np.max(np.abs(A_pcorr[0, 0] - A_pcorr[0, 0].T)))


# -----------------------------
# Save graphs (subject_id from filename)
# -----------------------------
def save_subject_graphs(
    out_dir: str | Path,
    subject_id: str,
    label: int,
    A_pcorr: np.ndarray,
    peak_windows: list[dict],
    ch_names: list[str],
    source_filename: str | None = None,
    source_filepath: str | None = None,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # compact peak window representation: [start, end, center]
    pw = np.array(
        [[w["start_samp"], w["end_samp"], w["center_samp"]] for w in peak_windows],
        dtype=np.int32
    )

    out_path = out_dir / f"{subject_id}_graphs.npz"
    np.savez_compressed(
        out_path,
        A_pcorr=A_pcorr.astype(np.float32),
        peak_windows=pw,
        label=np.int64(label),
        ch_names=np.array(ch_names, dtype=object),
        subject_id=np.array([subject_id], dtype=object),
        source_filename=np.array([source_filename if source_filename else ""], dtype=object),
        source_filepath=np.array([source_filepath if source_filepath else ""], dtype=object),
    )

    meta_path = out_dir / f"{subject_id}_graphs_meta.json"
    meta = {
        "subject_id": subject_id,
        "label": int(label),
        "A_pcorr_shape": list(A_pcorr.shape),
        "peak_windows_full": peak_windows,
        "ch_names": ch_names,
        "source_filename": source_filename,
        "source_filepath": source_filepath,
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", out_path)
    print("Saved:", meta_path)


# --- Your label and save path ---
label = 1  # dementia=1, HC=0
ch_names = epochs.ch_names

out_dir_path = Path(path_save) / "graphs_subjectwise/Dementia" ## change the path

save_subject_graphs(
    out_dir=out_dir_path,
    subject_id=subject_id,
    label=label,
    A_pcorr=A_pcorr,
    peak_windows=peak_windows,
    ch_names=ch_names,
    source_filename=filename,
    source_filepath=str(fn),
)


## Run the below code after completion of all subjects graphs (FTD and HC)

In [ ]:
## step 4: region-wise graph features from connectivity
ch_names= raw_eeg.ch_names
print(ch_names)
import numpy as np
import re, json
from pathlib import Path

# =========================
# CONFIG
# =========================
BASE_DIR = Path("/content/drive/My Drive/XXX/YYY/graphs_subjectwise/Dementia") ## change the file path

OVERWRITE = False   # True = overwrite the same file; False = create *_with_regions.npz next to it
AGG = "mean"        # "mean" or "median"

# If some saved files do not contain ch_names, we can fallback ONLY if N matches:
FALLBACK_CH_NAMES_19 = [
    "Fp1","Fp2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","T3","T4","T5","T6","Fz","Cz","Pz"
]

# =========================
# REGION DEFINITIONS
# =========================
REGIONS = ["frontal", "central", "parietal", "temporal", "occipital"]
REGION_TO_ID = {r: i for i, r in enumerate(REGIONS)}
N_REGIONS = len(REGIONS)

def _clean_ch_name(name: str) -> str:
    s = str(name).strip().upper()
    s = re.sub(r"[^A-Z0-9]", "", s)
    return s

def assign_region_5(name: str) -> str | None:
    """
    Your rule:
      F* -> frontal
      C* -> central
      P* -> parietal
      T* -> temporal
      O* -> occipital
    """
    ch = _clean_ch_name(name)
    if not ch:
        return None
    if ch.startswith("F"): return "frontal"
    if ch.startswith("C"): return "central"
    if ch.startswith("P"): return "parietal"
    if ch.startswith("T"): return "temporal"
    if ch.startswith("O"): return "occipital"
    return None

# =========================
# REGION FEATURE BUILDERS
# =========================
def compute_region_adjacency(A: np.ndarray, region_ids: np.ndarray,
                             n_regions: int = N_REGIONS, agg: str = "mean") -> np.ndarray:
    """
    A: (N, N) adjacency for ONE epoch & ONE peak
    region_ids: (N,) region id per channel
    Returns R: (5,5) region adjacency aggregated over channel pairs.
    """
    R = np.zeros((n_regions, n_regions), dtype=np.float32)
    idx_by_region = [np.where(region_ids == r)[0] for r in range(n_regions)]

    for p in range(n_regions):
        Ip = idx_by_region[p]
        for q in range(n_regions):
            Iq = idx_by_region[q]
            if Ip.size == 0 or Iq.size == 0:
                R[p, q] = 0.0
                continue

            block = A[np.ix_(Ip, Iq)]

            # within-region: exclude diagonal self-edges
            if p == q:
                if block.shape[0] <= 1:
                    R[p, q] = 0.0
                    continue
                mask = ~np.eye(block.shape[0], dtype=bool)
                vals = block[mask]
            else:
                vals = block.ravel()

            if vals.size == 0:
                R[p, q] = 0.0
            else:
                if agg == "mean":
                    R[p, q] = float(np.mean(vals))
                elif agg == "median":
                    R[p, q] = float(np.median(vals))
                else:
                    raise ValueError("agg must be 'mean' or 'median'")

    # symmetry safety
    R = 0.5 * (R + R.T)
    return R

def build_region_graphs(A_all: np.ndarray, region_ids: np.ndarray, agg: str = "mean") -> np.ndarray:
    """
    A_all: (n_epochs, n_peaks, N, N)
    Returns R_all: (n_epochs, n_peaks, 5, 5)
    """
    n_epochs, n_peaks, N, _ = A_all.shape
    R_all = np.zeros((n_epochs, n_peaks, N_REGIONS, N_REGIONS), dtype=np.float32)
    for n in range(n_epochs):
        for k in range(n_peaks):
            R_all[n, k] = compute_region_adjacency(A_all[n, k], region_ids, n_regions=N_REGIONS, agg=agg)
    return R_all

def build_region_pair_index(region_ids: np.ndarray, n_regions: int = N_REGIONS,
                            undirected: bool = True, contiguous: bool = True) -> np.ndarray:
    """
    Returns (N, N) region-pair id matrix.
    undirected=True -> (p,q) and (q,p) share id.
    contiguous=True -> compact ids in [0..R*(R+1)/2 - 1] for undirected pairs.
    """
    a = region_ids[:, None]
    b = region_ids[None, :]

    if undirected:
        p = np.minimum(a, b)
        q = np.maximum(a, b)
        if contiguous:
            start = p * n_regions - (p * (p - 1)) // 2
            pair_id = start + (q - p)
        else:
            pair_id = p * n_regions + q
    else:
        pair_id = a * n_regions + b

    return pair_id.astype(np.int64)

# =========================
# UPDATE ONE SUBJECT FILE
# =========================
def add_region_features_to_npz(npz_path: Path, overwrite: bool = False, agg: str = "mean") -> Path:
    with np.load(npz_path, allow_pickle=True) as data:
        # Find adjacency
        if "A_pcorr" in data.files:
            A_all = data["A_pcorr"]
            A_key = "A_pcorr"
        elif "A" in data.files:
            A_all = data["A"]
            A_key = "A"
        else:
            raise KeyError(f"{npz_path.name}: missing 'A_pcorr' or 'A'.")

        # Extract channel names
        if "ch_names" in data.files:
            ch_names = list(data["ch_names"])
        else:
            # fallback only if we can safely infer
            N = A_all.shape[-1]
            if N == 19:
                ch_names = FALLBACK_CH_NAMES_19
                print(f"[WARN] {npz_path.name}: 'ch_names' missing; using fallback 19-channel montage.")
            else:
                raise KeyError(f"{npz_path.name}: missing 'ch_names' and cannot infer for N={N}.")

        # Keep all existing arrays/fields
        existing = {k: data[k] for k in data.files}

    # Align region ids with ch_names order
    region_names = []
    unknown = []
    ch_to_region = {}
    for ch in ch_names:
        r = assign_region_5(ch)
        if r is None:
            unknown.append(str(ch))
            r = "central"  # fallback; or raise error if you prefer strict
        ch_to_region[str(ch)] = r
        region_names.append(r)

    region_ids = np.array([REGION_TO_ID[r] for r in region_names], dtype=np.int64)

    # Compute region adjacency per epoch/peak
    R_region = build_region_graphs(A_all, region_ids, agg=agg)  # (n_epochs, n_peaks, 5, 5)

    # Region-pair ids (for region_pair_embed)
    region_pair_id = build_region_pair_index(region_ids, undirected=True, contiguous=True)  # (N, N)

    # Edge-wise ids for upper-triangle edges (if you vectorize edges later)
    N = len(ch_names)
    triu_i, triu_j = np.triu_indices(N, k=1)
    edge_region_pair_id = region_pair_id[triu_i, triu_j]  # (E,)

    # Peak ids (0..T-1)
    n_peaks = A_all.shape[1]
    peak_ids = np.arange(n_peaks, dtype=np.int64)

    # Add region arrays
    existing.update({
        "R_region": R_region.astype(np.float32),
        "region_ids": region_ids,
        "region_pair_id": region_pair_id,
        "edge_region_pair_id": edge_region_pair_id,
        "peak_ids": peak_ids,
        "regions": np.array(REGIONS, dtype=object),
        "region_names_per_channel": np.array(region_names, dtype=object),
        "A_key_used": np.array([A_key], dtype=object),
    })

    out_path = npz_path if overwrite else npz_path.with_name(npz_path.stem + "_with_regions.npz")
    np.savez_compressed(out_path, **existing)

    # Save mapping as a readable sidecar JSON in the same folder
    json_path = out_path.with_suffix("").as_posix() + "_ch_to_region.json"
    with open(json_path, "w") as f:
        json.dump({
            "regions": REGIONS,
            "channel_to_region": ch_to_region,
            "unknown_channels_fallback_to_central": unknown
        }, f, indent=2)

    return out_path

# =========================
# RUN FOR ALL SUBJECT FILES IN THE FOLDER (SKIP IF OUTPUT ALREADY EXISTS)
# =========================
if not BASE_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {BASE_DIR}")

# Only take the base graph files (recommended)
npz_files = sorted(BASE_DIR.glob("*_graphs.npz"))

if len(npz_files) == 0:
    raise RuntimeError(f"No *_graphs.npz files found in: {BASE_DIR}")

print(f"Found {len(npz_files)} base graph files in: {BASE_DIR}")

updated = 0
skipped = 0
failed = 0

for f in npz_files:
    out_path = f.with_name(f.stem + "_with_regions.npz")

    # If output exists and overwrite is False -> skip
    if out_path.exists() and not OVERWRITE:
        print(f"[SKIP] {f.name} -> already exists: {out_path.name}")
        skipped += 1
        continue

    try:
        out_f = add_region_features_to_npz(f, overwrite=OVERWRITE, agg=AGG)
        print(f"[OK] {f.name}  ->  {out_f.name}")
        updated += 1
    except Exception as e:
        print(f"[FAIL] {f.name}: {e}")
        failed += 1

print(f"\nDone. Updated={updated}, Skipped={skipped}, Failed={failed}")
print("Outputs saved in the same folder:", BASE_DIR)


In [ ]:
# ============================================================
# Graph-memory gating + edge-memory refinement
#
# Hierarchical memory:
#   - Graph-level memory over region graphs (5x5)
#   - Edge-level refinement over channel-pair connectivity
#
# Compatibility:
#   Aliases GraphMemoryNetTF = GraphMemoryRefineNetTF at the end.
# ============================================================

import numpy as np
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
HEALTHY_DIR  = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity/graphs_subjectwise/Healthy") ## Change the path
DEMENTIA_DIR = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity/graphs_subjectwise/Dementia")
FILE_PATTERN = "*_with_regions.npz"

# ============================================================
# Helpers
# ============================================================
def _load_npz_keys(npz_path: Path) -> Dict[str, np.ndarray]:
    with np.load(npz_path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}

def _get_A_key(d: Dict[str, np.ndarray]) -> str:
    if "A_pcorr" in d:
        return "A_pcorr"
    if "A" in d:
        return "A"
    raise KeyError("Missing 'A_pcorr' or 'A' in npz.")

def build_epoch_index(
    healthy_dir: Path,
    dementia_dir: Path,
    pattern: str
) -> List[Tuple[str, int, int]]:
    """
    Returns list of samples: (npz_path_str, epoch_idx, label)
      label: 0 = Healthy, 1 = Dementia
    """
    samples: List[Tuple[str, int, int]] = []

    def collect(folder: Path, label: int):
        files = sorted(folder.glob(pattern))
        if len(files) == 0:
            raise RuntimeError(f"No files found in {folder} with pattern {pattern}")
        for f in files:
            d = _load_npz_keys(f)
            A_key = _get_A_key(d)
            A_all = d[A_key]  # (n_epochs, T, N, N)
            n_epochs = int(A_all.shape[0])
            for e in range(n_epochs):
                samples.append((str(f), e, label))

    collect(healthy_dir, label=0)
    collect(dementia_dir, label=1)
    return samples

def infer_constants_from_first_file(
    healthy_dir: Path,
    dementia_dir: Path,
    pattern: str
):
    """
    Infer:
      N nodes, E edges, region_pair_vocab size
    """
    files = sorted(list(healthy_dir.glob(pattern)) + list(dementia_dir.glob(pattern)))
    if len(files) == 0:
        raise RuntimeError("No npz files found to infer shapes.")
    d = _load_npz_keys(files[0])
    A_key = _get_A_key(d)
    A_all = d[A_key]
    N = int(A_all.shape[-1])
    edge_rpid = d["edge_region_pair_id"].astype(np.int64)
    E = int(edge_rpid.shape[0])
    vocab = int(edge_rpid.max()) + 1
    return N, E, vocab

# ============================================================
# tf.data dataset (epoch-level)
# ============================================================
def make_epoch_dataset(samples: List[Tuple[str, int, int]], N: int, E: int, shuffle: bool = True):
    """
    Each element yields:
      inputs = {
        "A": (T,N,N) float32,
        "R": (T,5,5) float32,
        "peak_mask": (T,) bool,
        "edge_rpid": (E,) int32
      }
      y = () float32
    """
    def gen():
        cache_path: Optional[str] = None
        cache_data: Optional[Dict[str, np.ndarray]] = None

        local_samples = list(samples)
        if shuffle:
            np.random.shuffle(local_samples)

        for path_str, epoch_idx, label in local_samples:
            if cache_path != path_str:
                cache_data = _load_npz_keys(Path(path_str))
                cache_path = path_str

            d = cache_data
            A_key = _get_A_key(d)
            A_all = d[A_key]                      # (n_epochs,T,N,N)
            R_all = d["R_region"]                 # (n_epochs,T,5,5)
            edge_rpid = d["edge_region_pair_id"]  # (E,)

            A = A_all[epoch_idx].astype(np.float32)
            R = R_all[epoch_idx].astype(np.float32)
            T = A.shape[0]
            peak_mask = np.ones((T,), dtype=np.bool_)

            assert A.shape[1] == N and A.shape[2] == N, f"N mismatch in {path_str}"
            assert edge_rpid.shape[0] == E, f"E mismatch in {path_str}"

            inputs = {
                "A": A,
                "R": R,
                "peak_mask": peak_mask,
                "edge_rpid": edge_rpid.astype(np.int32),
            }
            y = np.float32(label)
            yield inputs, y

    output_signature = (
        {
            "A": tf.TensorSpec(shape=(None, N, N), dtype=tf.float32),
            "R": tf.TensorSpec(shape=(None, 5, 5), dtype=tf.float32),
            "peak_mask": tf.TensorSpec(shape=(None,), dtype=tf.bool),
            "edge_rpid": tf.TensorSpec(shape=(E,), dtype=tf.int32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(samples)), reshuffle_each_iteration=True)

    return ds

# ============================================================
# Stats helpers
# ============================================================
def _quantile_approx(x: tf.Tensor, q: float) -> tf.Tensor:
    """
    x: (..., E)
    returns approx quantile along last axis
    """
    x_sorted = tf.sort(x, axis=-1)
    n = tf.shape(x_sorted)[-1]
    idx = tf.cast(
        tf.round(tf.cast(n - 1, tf.float32) * tf.constant(q, tf.float32)),
        tf.int32
    )
    return tf.gather(x_sorted, idx, axis=-1)

def edge_stats(x_edges: tf.Tensor) -> tf.Tensor:
    """
    x_edges: (B,E)
    returns stats: (B,6)
    """
    mean = tf.reduce_mean(x_edges, axis=-1)
    std  = tf.math.reduce_std(x_edges, axis=-1)
    med  = _quantile_approx(x_edges, 0.50)
    q25  = _quantile_approx(x_edges, 0.25)
    q75  = _quantile_approx(x_edges, 0.75)
    mx   = tf.reduce_max(x_edges, axis=-1)
    return tf.stack([mean, std, med, q25, q75, mx], axis=-1)

# ============================================================
# Gather / scatter helpers
# ============================================================
def batch_gather_edges(A: tf.Tensor, edge_i: tf.Tensor, edge_j: tf.Tensor) -> tf.Tensor:
    """
    A: (B,N,N)
    edge_i, edge_j: (E,)
    returns: (B,E)
    """
    B = tf.shape(A)[0]
    E = tf.shape(edge_i)[0]

    b_idx = tf.reshape(tf.range(B, dtype=tf.int32), (B, 1, 1))
    b_idx = tf.tile(b_idx, (1, E, 1))  # (B,E,1)

    i = tf.reshape(edge_i, (1, E, 1))
    j = tf.reshape(edge_j, (1, E, 1))
    i = tf.tile(i, (B, 1, 1))
    j = tf.tile(j, (B, 1, 1))

    idx = tf.concat([b_idx, i, j], axis=-1)  # (B,E,3)
    return tf.gather_nd(A, idx)              # (B,E)

def scatter_symmetric_edges(edge_i: tf.Tensor, edge_j: tf.Tensor, H_edges: tf.Tensor, N: int) -> tf.Tensor:
    """
    Build symmetric (B,N,N) matrix from upper-tri edges.
    H_edges: (B,E)
    """
    Bsz = tf.shape(H_edges)[0]
    E = tf.shape(edge_i)[0]

    b_idx = tf.reshape(tf.range(Bsz, dtype=tf.int32), (Bsz, 1, 1))
    b_idx = tf.tile(b_idx, (1, E, 1))  # (B,E,1)

    i = tf.reshape(edge_i, (1, E, 1)); i = tf.tile(i, (Bsz, 1, 1))
    j = tf.reshape(edge_j, (1, E, 1)); j = tf.tile(j, (Bsz, 1, 1))

    idx_ij = tf.concat([b_idx, i, j], axis=-1)  # (B,E,3)
    idx_ji = tf.concat([b_idx, j, i], axis=-1)  # (B,E,3)

    idx_all = tf.concat(
        [tf.reshape(idx_ij, (-1, 3)), tf.reshape(idx_ji, (-1, 3))],
        axis=0
    )

    upd = tf.reshape(H_edges, (-1,))
    upd_all = tf.concat([upd, upd], axis=0)

    H0 = tf.zeros((Bsz, N, N), dtype=H_edges.dtype)
    H = tf.tensor_scatter_nd_update(H0, idx_all, upd_all)
    return H

# ============================================================
# Region-pair lookup
# IMPORTANT:
#   Must match Step 4 build_region_pair_index(..., undirected=True, contiguous=True)
# ============================================================
def build_region_pair_lookup_contiguous(n_regions: int = 5):
    ii, jj = [], []
    for p in range(n_regions):
        for q in range(p, n_regions):
            ii.append(p)
            jj.append(q)
    return np.array(ii, dtype=np.int32), np.array(jj, dtype=np.int32)

# ============================================================
# Step 5.1 ThresholdNet
# ============================================================
class ThresholdNet(layers.Layer):
    def __init__(self, peak_emb_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
        ])
        self.out_theta = layers.Dense(1)
        self.out_tau   = layers.Dense(1)

    def call(self, stats_a, stats_r, peak_emb):
        x = tf.concat([stats_a, stats_r, peak_emb], axis=-1)
        h = self.mlp(x)
        theta = self.out_theta(h)
        tau = tf.nn.softplus(self.out_tau(h)) + 1e-6
        return theta, tau

# ============================================================
# Graph-memory gating network
# ============================================================
class GraphGateNet(layers.Layer):
    """
    Learns graph-level gates over region graph memory.
    Outputs:
      I_g, F_g, O_g, S_g
    where
      S_g controls how much to favor subset-merge vs similarity-average candidate graph.
    """
    def __init__(self, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
            layers.Dense(4, activation=None),
        ])

    def call(self, x_graph):
        z = self.mlp(x_graph)
        I_g = tf.sigmoid(z[..., 0])   # graph write gate
        F_g = tf.sigmoid(z[..., 1])   # graph keep/forget gate
        O_g = tf.sigmoid(z[..., 2])   # graph output gate
        S_g = tf.sigmoid(z[..., 3])   # subset-vs-sim candidate blend
        return I_g, F_g, O_g, S_g

# ============================================================
# Edge-memory refinement network
# ============================================================
class EdgeRefineNet(layers.Layer):
    """
    Refinement gates for channel edges, conditioned on graph memory.
    Candidate is sigmoid so edge memory stays nonnegative and compatible
    with connectivity weights.
    """
    def __init__(self, edge_feat_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
            layers.Dense(4, activation=None)   # I_e, F_e, O_e, Ghat_e
        ])

    def call(self, x_edge):
        z = self.mlp(x_edge)
        I_e = tf.sigmoid(z[..., 0])
        F_e = tf.sigmoid(z[..., 1])
        O_e = tf.sigmoid(z[..., 2])
        Ghat_e = tf.sigmoid(z[..., 3])  # keep nonnegative
        return I_e, F_e, O_e, Ghat_e

# ============================================================
# Step 5.3 Shared GNN
# ============================================================
class SimpleGCN(layers.Layer):
    """
    Residual GCN encoder
    """
    def __init__(
        self,
        n_nodes: int,
        node_emb_dim: int = 64,
        hidden_dim: int = 64,
        out_dim: int = 128,
        num_layers: int = 3,
        dropout_rate: float = 0.3,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_nodes = n_nodes
        self.node_emb = layers.Embedding(input_dim=n_nodes, output_dim=node_emb_dim)

        self.input_proj = None
        if node_emb_dim != hidden_dim:
            self.input_proj = layers.Dense(hidden_dim, activation="relu")

        self.gcn_layers = [layers.Dense(hidden_dim, activation="relu") for _ in range(num_layers)]
        self.final_layer = layers.Dense(out_dim, activation="relu")

        self.dropout = layers.Dropout(dropout_rate)
        self.node_idx = tf.constant(np.arange(n_nodes, dtype=np.int32))

    def call(self, A, training=False):
        B = tf.shape(A)[0]
        N = self.n_nodes

        X = self.node_emb(self.node_idx)              # (N,D0)
        X = tf.tile(X[None, :, :], [B, 1, 1])        # (B,N,D0)

        if self.input_proj is not None:
            X = self.input_proj(X)

        I = tf.eye(N, batch_shape=[B], dtype=A.dtype)
        A_hat = A + I

        D = tf.reduce_sum(A_hat, axis=-1)
        D_inv_sqrt = tf.pow(tf.maximum(D, 1e-6), -0.5)
        A_norm = (D_inv_sqrt[..., None] * A_hat) * (D_inv_sqrt[:, None, :])

        for dense in self.gcn_layers:
            H = tf.matmul(A_norm, X)
            H = dense(H)
            H = self.dropout(H, training=training)
            X = X + H

        H = tf.matmul(A_norm, X)
        H = self.final_layer(H)
        H = self.dropout(H, training=training)

        z = tf.reduce_mean(H, axis=1)
        return z

# ============================================================
# Step 5.4 Peak attention pooling
# ============================================================
class PeakAttentionPool(layers.Layer):
    def __init__(self, z_dim: int, peak_emb_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.score = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(1, activation=None),
        ])
        self.z_dim = z_dim
        self.peak_emb_dim = peak_emb_dim

    def call(self, Z, peak_emb_seq, peak_mask):
        x = tf.concat([Z, peak_emb_seq], axis=-1)
        s = tf.squeeze(self.score(x), axis=-1)

        neg_inf = tf.constant(-1e9, dtype=s.dtype)
        s_masked = tf.where(peak_mask, s, neg_inf)
        beta = tf.nn.softmax(s_masked, axis=-1)

        z_pool = tf.reduce_sum(beta[..., None] * Z, axis=1)
        return z_pool, beta

# ============================================================
# Full graph-memory gating + edge-memory refinement model
# ============================================================
class GraphMemoryRefineNetTF(keras.Model):
    """
    Hierarchical recurrence:

      Step 5.1:
        ThresholdNet predicts (theta_k, tau_k) and builds Abar_k

      Step 5.2a:
        Graph-memory gating over region graphs:
          C_graph(k-1), H_graph(k-1) -> C_graph(k), H_graph(k)

      Step 5.2b:
        Edge-memory refinement conditioned on graph memory:
          C_edge(k-1), H_graph(k) -> C_edge(k), H_edge(k)

      Step 5.3:
        Shared GNN over refined hidden graph H_k

      Step 5.4:
        Attention pooling over peaks + classifier

    NOTE:
      Graph memory is region-level (5x5), so subset/similarity is region-level.
      Exact electrode-level subset matching is not represented here.
    """
    def __init__(
        self,
        n_nodes: int,
        region_pair_vocab: int,
        max_peaks: int = 64,

        # compatibility args (optional / ignored)
        n_edges: Optional[int] = None,
        d_model: Optional[int] = None,
        r_emb_dim: Optional[int] = None,
        edge_emb_dim: Optional[int] = None,
        summary_dim: Optional[int] = None,
        num_heads: Optional[int] = None,
        ff_dim: Optional[int] = None,
        dropout: Optional[float] = None,
        l2_reg: Optional[float] = None,

        # actual model params
        region_pair_emb_dim: int = 8,
        peak_emb_dim: int = 8,
        thr_hidden: int = 64,
        graph_gate_hidden: int = 64,
        edge_gate_hidden: int = 64,
        gnn_out_dim: int = 128,
        gnn_num_layers: int = 3,
        gnn_dropout: float = 0.3,
        clf_hidden1: int = 128,
        clf_hidden2: int = 64,
        clf_dropout: float = 0.2,

        # region matching hyperparameters
        region_presence_theta: float = 0.15,
        region_subset_thr: float = 0.85,
        region_sim_thr: float = 0.60,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_nodes = int(n_nodes)
        self.max_peaks = int(max_peaks)
        self.region_presence_theta = float(region_presence_theta)
        self.region_subset_thr = float(region_subset_thr)
        self.region_sim_thr = float(region_sim_thr)

        # ----------------------------------------------------
        # channel-edge indices (upper triangle)
        # ----------------------------------------------------
        edge_i, edge_j = np.triu_indices(self.n_nodes, k=1)
        self.edge_i = tf.constant(edge_i.astype(np.int32))
        self.edge_j = tf.constant(edge_j.astype(np.int32))
        self.E = int(edge_i.shape[0])

        if n_edges is not None and int(n_edges) != self.E:
            raise ValueError(
                f"n_edges={int(n_edges)} does not match upper-triangle edge count "
                f"E={self.E} for n_nodes={self.n_nodes}"
            )

        # ----------------------------------------------------
        # region-pair lookup for Step 4 contiguous ids
        # full pairs: include diagonal (15 ids)
        # ----------------------------------------------------
        reg_all_i, reg_all_j = build_region_pair_lookup_contiguous(5)
        self.reg_all_i = tf.constant(reg_all_i, dtype=tf.int32)  # (15,)
        self.reg_all_j = tf.constant(reg_all_j, dtype=tf.int32)  # (15,)

        # off-diagonal region graph edges only (10 inter-region edges)
        reg_match_i, reg_match_j = np.triu_indices(5, k=1)
        self.reg_match_i = tf.constant(reg_match_i.astype(np.int32))  # (10,)
        self.reg_match_j = tf.constant(reg_match_j.astype(np.int32))  # (10,)

        # embeddings
        self.peak_emb = layers.Embedding(input_dim=self.max_peaks, output_dim=peak_emb_dim)
        self.region_pair_emb = layers.Embedding(input_dim=int(region_pair_vocab), output_dim=region_pair_emb_dim)

        # Step 5.1
        self.threshold_net = ThresholdNet(peak_emb_dim=peak_emb_dim, hidden=thr_hidden)

        # Step 5.2a graph-memory gating
        self.graph_gate = GraphGateNet(hidden=graph_gate_hidden)

        # Step 5.2b edge-memory refinement
        # scalar features:
        #   Abar_edge, C_edge_prev, |Abar-C_edge_prev|,
        #   graph_prior_edge, |graph_prior_edge-C_edge_prev|, |Abar-graph_prior_edge|,
        #   graph_update_gate, subset_score, jaccard_score
        # => 9 scalar features
        edge_feat_dim = 9 + region_pair_emb_dim + peak_emb_dim
        self.edge_refine = EdgeRefineNet(edge_feat_dim=edge_feat_dim, hidden=edge_gate_hidden)

        # Step 5.3
        self.gnn = SimpleGCN(
            n_nodes=self.n_nodes,
            node_emb_dim=64,
            hidden_dim=64,
            out_dim=gnn_out_dim,
            num_layers=gnn_num_layers,
            dropout_rate=gnn_dropout
        )

        # Step 5.4
        self.peak_pool = PeakAttentionPool(z_dim=gnn_out_dim, peak_emb_dim=peak_emb_dim)

        # classifier (returns probability)
        self.classifier = keras.Sequential([
            layers.Dense(clf_hidden1, activation="relu"),
            layers.Dropout(clf_dropout),
            layers.Dense(clf_hidden2, activation="relu"),
            layers.Dropout(clf_dropout),
            layers.Dense(1, activation="sigmoid")
        ])

        # interpretability
        self.last_beta = None
        self.last_theta = None
        self.last_tau = None
        self.last_graph_subset = None
        self.last_graph_jaccard = None
        self.last_graph_update = None

    def _region_match_gate(self, C_graph_prev, R_k, mask_b):
        """
        Region-level match between previous graph memory and current region graph.

        Uses only OFF-DIAGONAL inter-region edges (10 edges).

        Returns:
          update_gate : (B,)
          subset_score: (B,)
          jaccard     : (B,)
          subset_gate : (B,)
        """
        prev_pairs = batch_gather_edges(C_graph_prev, self.reg_match_i, self.reg_match_j)  # (B,10)
        curr_pairs = batch_gather_edges(R_k,          self.reg_match_i, self.reg_match_j)   # (B,10)

        prev_active_b = prev_pairs >= self.region_presence_theta
        curr_active_b = curr_pairs >= self.region_presence_theta

        inter = tf.reduce_sum(tf.cast(tf.logical_and(prev_active_b, curr_active_b), tf.float32), axis=-1)
        prev_count = tf.reduce_sum(tf.cast(prev_active_b, tf.float32), axis=-1)
        union = tf.reduce_sum(tf.cast(tf.logical_or(prev_active_b, curr_active_b), tf.float32), axis=-1)

        subset_score = tf.where(prev_count > 0.0, inter / (prev_count + 1e-6), tf.zeros_like(inter))
        jaccard = tf.where(union > 0.0, inter / (union + 1e-6), tf.zeros_like(inter))

        subset_gate = tf.cast(subset_score >= self.region_subset_thr, tf.float32)
        sim_gate = tf.cast(jaccard >= self.region_sim_thr, tf.float32)
        update_gate = tf.maximum(subset_gate, sim_gate)

        update_gate = mask_b * update_gate
        subset_gate = mask_b * subset_gate
        subset_score = mask_b * subset_score
        jaccard = mask_b * jaccard

        return update_gate, subset_score, jaccard, subset_gate

    def call(self, inputs, training=False):
        """
        inputs:
          A: (B,T,N,N)
          R: (B,T,5,5)
          peak_mask: (B,T) bool
          edge_rpid: (B,E) or (E,)

        returns:
          probabilities: (B,)
        """
        A = inputs["A"]
        R = inputs["R"]
        peak_mask = inputs["peak_mask"]
        edge_rpid = inputs["edge_rpid"]

        B = tf.shape(A)[0]
        T = tf.shape(A)[1]
        N = self.n_nodes

        # edge_rpid may be batched
        if edge_rpid.shape.rank == 2:
            edge_rpid_1d = tf.cast(edge_rpid[0], tf.int32)  # (E,)
        else:
            edge_rpid_1d = tf.cast(edge_rpid, tf.int32)

        # edge region-pair embedding
        rp_emb = self.region_pair_emb(edge_rpid_1d)  # (E,drp)

        # map each channel-edge region_pair_id -> region pair indices
        edge_reg_i = tf.gather(self.reg_all_i, edge_rpid_1d)  # (E,)
        edge_reg_j = tf.gather(self.reg_all_j, edge_rpid_1d)  # (E,)

        # peak embeddings
        peak_ids = tf.range(T, dtype=tf.int32)
        peak_ids = tf.minimum(peak_ids, tf.constant(self.max_peaks - 1, tf.int32))
        peak_emb_seq = self.peak_emb(peak_ids)  # (T,dpeak)
        peak_emb_seq_B = tf.tile(peak_emb_seq[None, :, :], [B, 1, 1])  # (B,T,dpeak)

        # ----------------------------------------------------
        # hierarchical memory states
        # C_graph : (B,5,5) graph-level region memory
        # C_edge  : (B,E)   edge-level refinement memory
        # ----------------------------------------------------
        C_graph = tf.zeros((B, 5, 5), dtype=tf.float32)
        C_edge  = tf.zeros((B, self.E), dtype=tf.float32)

        Z_list = tf.TensorArray(tf.float32, size=T)
        theta_list = tf.TensorArray(tf.float32, size=T)
        tau_list = tf.TensorArray(tf.float32, size=T)
        subset_list = tf.TensorArray(tf.float32, size=T)
        jaccard_list = tf.TensorArray(tf.float32, size=T)
        update_list = tf.TensorArray(tf.float32, size=T)

        def loop_body(k, C_graph, C_edge, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list):
            A_k = A[:, k, :, :]  # (B,N,N)
            R_k = R[:, k, :, :]  # (B,5,5)

            mask_b = tf.cast(peak_mask[:, k], tf.float32)   # (B,)
            mask_edge = mask_b[:, None]                     # (B,1)

            # safety for region graph
            R_k = 0.5 * (R_k + tf.transpose(R_k, perm=[0, 2, 1]))
            R_k = tf.clip_by_value(R_k, 0.0, 1.0)
            R_k = mask_b[:, None, None] * R_k

            # ------------------------------------------------
            # stats over channel edges
            # ------------------------------------------------
            A_edges = batch_gather_edges(A_k, self.edge_i, self.edge_j)   # (B,E)
            stats_a = edge_stats(A_edges)                                 # (B,6)

            # ------------------------------------------------
            # stats over current region inter-region edges
            # ------------------------------------------------
            R_edges = batch_gather_edges(R_k, self.reg_match_i, self.reg_match_j)  # (B,10)
            stats_r = edge_stats(R_edges)                                           # (B,6)

            pk_emb = peak_emb_seq_B[:, k, :]   # (B,dpeak)

            # ------------------------------------------------
            # Step 5.1 ThresholdNet
            # ------------------------------------------------
            theta_k, tau_k = self.threshold_net(stats_a, stats_r, pk_emb)
            theta_k = tf.squeeze(theta_k, axis=-1)
            tau_k   = tf.squeeze(tau_k, axis=-1)

            theta_k = mask_b * theta_k
            tau_k   = mask_b * tau_k + (1.0 - mask_b) * 1.0

            theta_mat = theta_k[:, None, None]
            tau_mat   = tau_k[:, None, None]

            M_k = tf.sigmoid((A_k - theta_mat) / tau_mat)
            M_k = mask_b[:, None, None] * M_k
            Abar_k = M_k * A_k
            Abar_k = tf.linalg.set_diag(Abar_k, tf.zeros((B, N), dtype=Abar_k.dtype))

            # ------------------------------------------------
            # Step 5.2a Graph-memory gating
            # ------------------------------------------------
            def first_peak_branch():
                ones = mask_b
                C_graph_new = R_k
                H_graph = R_k
                return ones, ones, ones, ones, C_graph_new, H_graph

            def later_peak_branch():
                graph_update_gate, subset_score, jaccard_score, subset_gate = self._region_match_gate(C_graph, R_k, mask_b)

                prev_pairs = batch_gather_edges(C_graph, self.reg_match_i, self.reg_match_j)  # (B,10)
                curr_pairs = batch_gather_edges(R_k,     self.reg_match_i, self.reg_match_j)  # (B,10)
                stats_prev = edge_stats(prev_pairs)                                            # (B,6)
                stats_curr = edge_stats(curr_pairs)                                            # (B,6)
                stats_diff = tf.abs(stats_curr - stats_prev)                                   # (B,6)

                x_graph = tf.concat(
                    [
                        stats_prev,
                        stats_curr,
                        stats_diff,
                        subset_score[:, None],
                        jaccard_score[:, None],
                        pk_emb
                    ],
                    axis=-1
                )

                I_g, F_g, O_g, S_g = self.graph_gate(x_graph)   # each (B,)

                # candidate graph:
                #   subset-like candidate -> max(previous, current)
                #   similar-like candidate -> average(previous, current)
                R_subset = tf.maximum(C_graph, R_k)
                R_sim = 0.5 * (C_graph + R_k)

                Sg_mat = S_g[:, None, None]
                I_mat  = I_g[:, None, None]
                F_mat  = F_g[:, None, None]
                O_mat  = O_g[:, None, None]
                Ug_mat = graph_update_gate[:, None, None]

                graph_cand = Sg_mat * R_subset + (1.0 - Sg_mat) * R_sim
                C_graph_prop = F_mat * C_graph + I_mat * graph_cand
                C_graph_prop = tf.clip_by_value(C_graph_prop, 0.0, 1.0)

                C_graph_new = Ug_mat * C_graph_prop + (1.0 - Ug_mat) * C_graph
                C_graph_new = tf.clip_by_value(C_graph_new, 0.0, 1.0)

                H_graph = O_mat * C_graph_new
                H_graph = tf.clip_by_value(H_graph, 0.0, 1.0)

                return graph_update_gate, subset_score, jaccard_score, subset_gate, C_graph_new, H_graph

            graph_update_gate, subset_score, jaccard_score, subset_gate, C_graph_new, H_graph = tf.cond(
                tf.equal(k, 0),
                first_peak_branch,
                later_peak_branch
            )

            # ------------------------------------------------
            # Step 5.2b Edge-memory refinement
            # Graph memory provides coarse edge prior
            # ------------------------------------------------
            Abar_edges = batch_gather_edges(Abar_k, self.edge_i, self.edge_j)  # (B,E)
            diff_prev = tf.abs(Abar_edges - C_edge)                            # (B,E)

            graph_prior_edge = batch_gather_edges(H_graph, edge_reg_i, edge_reg_j)  # (B,E)
            diff_mem_graph = tf.abs(graph_prior_edge - C_edge)                       # (B,E)
            diff_input_graph = tf.abs(Abar_edges - graph_prior_edge)                 # (B,E)

            rp = tf.tile(rp_emb[None, :, :], [B, 1, 1])                               # (B,E,drp)
            pk = tf.tile(pk_emb[:, None, :], [1, self.E, 1])                          # (B,E,dpeak)

            graph_update_e = tf.tile(graph_update_gate[:, None, None], [1, self.E, 1])
            subset_e       = tf.tile(subset_score[:, None, None],      [1, self.E, 1])
            jaccard_e      = tf.tile(jaccard_score[:, None, None],     [1, self.E, 1])

            x_edge = tf.concat([
                Abar_edges[..., None],          # current masked channel edge
                C_edge[..., None],              # previous edge memory
                diff_prev[..., None],           # input-vs-previous edge diff
                graph_prior_edge[..., None],    # graph-memory prior for this channel edge
                diff_mem_graph[..., None],      # memory-vs-graph diff
                diff_input_graph[..., None],    # input-vs-graph diff
                graph_update_e,                 # graph update gate
                subset_e,                       # subset score
                jaccard_e,                      # similarity score
                rp,                             # region-pair embedding
                pk                              # peak embedding
            ], axis=-1)

            I_e, F_e, O_e, Ghat_e = self.edge_refine(x_edge)

            # edge proposal
            C_edge_prop = F_e * C_edge + I_e * Ghat_e
            C_edge_prop = tf.clip_by_value(C_edge_prop, 0.0, 1.0)

            # refinement around graph prior
            refine_alpha = graph_update_gate[:, None] * tf.clip_by_value(
                0.5 * subset_score[:, None] + 0.5 * jaccard_score[:, None],
                0.0, 1.0
            )

            C_edge_refined = refine_alpha * graph_prior_edge + (1.0 - refine_alpha) * C_edge_prop
            C_edge_refined = tf.clip_by_value(C_edge_refined, 0.0, 1.0)

            # if graph does not match, keep previous edge memory
            C_edge_new = graph_update_gate[:, None] * C_edge_refined + (1.0 - graph_update_gate[:, None]) * C_edge
            C_edge_new = tf.clip_by_value(C_edge_new, 0.0, 1.0)

            H_prev = C_edge
            H_prop = O_e * C_edge_new
            H_edges = graph_update_gate[:, None] * H_prop + (1.0 - graph_update_gate[:, None]) * H_prev
            H_edges = tf.clip_by_value(H_edges, 0.0, 1.0)

            # padded peaks -> keep previous
            C_edge_new = mask_edge * C_edge_new + (1.0 - mask_edge) * C_edge
            H_edges    = mask_edge * H_edges    + (1.0 - mask_edge) * H_prev
            C_graph_new = mask_b[:, None, None] * C_graph_new + (1.0 - mask_b[:, None, None]) * C_graph

            # build refined hidden channel graph H_k
            H_mat = scatter_symmetric_edges(self.edge_i, self.edge_j, H_edges, N)
            H_mat = tf.clip_by_value(H_mat, 0.0, 1.0)

            # ------------------------------------------------
            # Step 5.3 Shared GNN
            # ------------------------------------------------
            z_k = self.gnn(H_mat, training=training)
            z_k = z_k * mask_b[:, None]

            Z_list = Z_list.write(k, z_k)
            theta_list = theta_list.write(k, theta_k)
            tau_list = tau_list.write(k, tau_k)
            subset_list = subset_list.write(k, subset_score)
            jaccard_list = jaccard_list.write(k, jaccard_score)
            update_list = update_list.write(k, graph_update_gate)

            return (
                k + 1,
                C_graph_new,
                C_edge_new,
                Z_list,
                theta_list,
                tau_list,
                subset_list,
                jaccard_list,
                update_list
            )

        def loop_cond(k, C_graph, C_edge, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list):
            return k < T

        k0 = tf.constant(0, dtype=tf.int32)
        _, C_graph_final, C_edge_final, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list = tf.while_loop(
            loop_cond,
            loop_body,
            loop_vars=[
                k0,
                C_graph,
                C_edge,
                Z_list,
                theta_list,
                tau_list,
                subset_list,
                jaccard_list,
                update_list
            ],
            parallel_iterations=1
        )

        Z = tf.transpose(Z_list.stack(), perm=[1, 0, 2])  # (B,T,d)

        # ----------------------------------------------------
        # Step 5.4 Peak attention pooling
        # ----------------------------------------------------
        z_pool, beta = self.peak_pool(Z, peak_emb_seq_B, peak_mask)

        probs = tf.squeeze(self.classifier(z_pool, training=training), axis=-1)

        # store for analysis
        self.last_beta = beta
        self.last_theta = tf.transpose(theta_list.stack(), perm=[1, 0])         # (B,T)
        self.last_tau   = tf.transpose(tau_list.stack(),   perm=[1, 0])         # (B,T)
        self.last_graph_subset  = tf.transpose(subset_list.stack(),  perm=[1, 0])   # (B,T)
        self.last_graph_jaccard = tf.transpose(jaccard_list.stack(), perm=[1, 0])   # (B,T)
        self.last_graph_update  = tf.transpose(update_list.stack(),  perm=[1, 0])   # (B,T)

        return probs

# keep old name for compatibility with training code
GraphMemoryNetTF = GraphMemoryRefineNetTF

# ============================================================
# Sanity check
# ============================================================
samples = build_epoch_index(HEALTHY_DIR, DEMENTIA_DIR, FILE_PATTERN)
N, E, vocab = infer_constants_from_first_file(HEALTHY_DIR, DEMENTIA_DIR, FILE_PATTERN)

print("Total epoch samples:", len(samples))
print("N nodes:", N, "E edges:", E, "region_pair_vocab:", vocab)

ds = make_epoch_dataset(samples, N=N, E=E, shuffle=True)

BATCH_SIZE = 8
ds = ds.padded_batch(
    BATCH_SIZE,
    padded_shapes=(
        {
            "A": (None, N, N),
            "R": (None, 5, 5),
            "peak_mask": (None,),
            "edge_rpid": (E,),
        },
        ()
    ),
    padding_values=(
        {
            "A": tf.constant(0.0, tf.float32),
            "R": tf.constant(0.0, tf.float32),
            "peak_mask": tf.constant(False, tf.bool),
            "edge_rpid": tf.constant(0, tf.int32),
        },
        tf.constant(0.0, tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

model = GraphMemoryRefineNetTF(
    n_nodes=N,
    region_pair_vocab=vocab,
    max_peaks=64,
    region_pair_emb_dim=8,
    peak_emb_dim=8,
    thr_hidden=64,
    graph_gate_hidden=64,
    edge_gate_hidden=64,
    gnn_out_dim=128,
    gnn_num_layers=3,
    gnn_dropout=0.3,
    clf_hidden1=128,
    clf_hidden2=64,
    clf_dropout=0.2,
    region_presence_theta=0.15,
    region_subset_thr=0.85,
    region_sim_thr=0.60,
)

batch_inputs, batch_y = next(iter(ds))
probs = model(batch_inputs, training=False)

print("output shape:", probs.shape)
print("beta shape:", model.last_beta.shape if model.last_beta is not None else None)
print("theta shape:", model.last_theta.shape if model.last_theta is not None else None)
print("tau shape:", model.last_tau.shape if model.last_tau is not None else None)
print("graph subset shape:", model.last_graph_subset.shape if model.last_graph_subset is not None else None)
print("graph jaccard shape:", model.last_graph_jaccard.shape if model.last_graph_jaccard is not None else None)
print("graph update shape:", model.last_graph_update.shape if model.last_graph_update is not None else None)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.5)]
)

print("Model compiled successfully.")

In [ ]:
## FINAL HiGMR-NET Model (Dementia-only, FAST CACHED VERSION)
# ============================================================
# This version:
#   1) avoids KeyError when edge_region_pair_id / R_region are missing
#   2) builds missing metadata ONCE per file and caches it
#   3) makes the sanity-check fast
#   4) explains why no training starts in dementia-only mode
#
# IMPORTANT:
#   - This is NOT a binary classification setup, because all labels = 1
#   - So this cell is for sanity-check / forward pass / feature extraction only
#   - For real classification training, use both HC and Dementia folders
# ============================================================

import numpy as np
import mne
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DEMENTIA_DIR = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity/graphs_subjectwise/Dementia")
FILE_PATTERN = "*_graphs.npz"

N_REGIONS = 5
REGION_NAMES = ["frontal", "central", "parietal", "temporal", "occipital"]

# Default 19-channel HC order if a 19-channel file has no channel names
HC_19_CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T3", "C3", "Cz", "C4", "T4", "T5", "P3",
    "Pz", "P4", "T6", "O1", "O2"
]

# region ids for the above 19-channel order:
# frontal=0, central=1, parietal=2, temporal=3, occipital=4
DEFAULT_19_REGION_IDS = np.array([
    0, 0, 0, 0, 0, 0, 0,   # Fp1 Fp2 F7 F3 Fz F4 F8
    3,                     # T3
    1, 1, 1,               # C3 Cz C4
    3, 3,                  # T4 T5
    2, 2, 2,               # P3 Pz P4
    3,                     # T6
    4, 4                   # O1 O2
], dtype=np.int64)

# ============================================================
# Helpers
# ============================================================
def _load_npz_keys(npz_path: Path) -> Dict[str, np.ndarray]:
    with np.load(npz_path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}

def _get_A_key(d: Dict[str, np.ndarray]) -> str:
    if "A_pcorr" in d:
        return "A_pcorr"
    if "A" in d:
        return "A"
    raise KeyError("Missing 'A_pcorr' or 'A' in npz.")

def _normalize_name(s: str) -> str:
    return str(s).strip().upper().replace(".", "").replace(" ", "")

def _canonicalize_1020_name(name: str) -> str:
    s = _normalize_name(name)
    alias = {
        "T3": "T7",
        "T4": "T8",
        "T5": "P7",
        "T6": "P8",
    }
    return alias.get(s, s)

TARGET_19_CANONICAL = [_canonicalize_1020_name(ch) for ch in HC_19_CHANNELS]

def preprocess_A_np(A: np.ndarray) -> np.ndarray:
    A = np.nan_to_num(A, nan=0.0, posinf=1.0, neginf=-1.0).astype(np.float32)
    A = 0.5 * (A + np.swapaxes(A, -1, -2))

    idx = np.arange(A.shape[-1])
    A[..., idx, idx] = 0.0

    if np.nanmin(A) < 0.0:
        A = np.clip(A, -1.0, 1.0)
        A = 0.5 * (A + 1.0)
    else:
        A = np.clip(A, 0.0, 1.0)

    A[..., idx, idx] = 0.0
    return A.astype(np.float32)

def preprocess_R_np(R: np.ndarray) -> np.ndarray:
    R = np.nan_to_num(R, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)
    R = 0.5 * (R + np.swapaxes(R, -1, -2))
    R = np.clip(R, 0.0, 1.0)
    return R.astype(np.float32)

# ============================================================
# Channel-name / region-id inference
# ============================================================
def extract_channel_names_from_npz(d: Dict[str, np.ndarray]) -> Optional[List[str]]:
    candidate_keys = [
        "ch_names", "channel_names", "channels", "chan_names",
        "labels", "channel_labels", "electrode_names", "electrodes"
    ]

    for k in candidate_keys:
        if k in d:
            try:
                arr = np.asarray(d[k])
                if arr.ndim == 0:
                    continue
                names = [str(x) for x in arr.ravel().tolist()]
                if len(names) > 0:
                    return names
            except Exception:
                pass

    if "chanlocs" in d:
        chanlocs = d["chanlocs"]

        try:
            if hasattr(chanlocs, "dtype") and chanlocs.dtype.names is not None:
                if "labels" in chanlocs.dtype.names:
                    names = [str(x) for x in chanlocs["labels"].ravel().tolist()]
                    if len(names) > 0:
                        return names
        except Exception:
            pass

        try:
            flat = np.asarray(chanlocs, dtype=object).ravel().tolist()
            names = []
            for x in flat:
                if isinstance(x, dict) and "labels" in x:
                    names.append(str(x["labels"]))
                elif hasattr(x, "labels"):
                    names.append(str(x.labels))
                elif hasattr(x, "label"):
                    names.append(str(x.label))
            if len(names) > 0:
                return names
        except Exception:
            pass

    return None

def _build_biosemi128_name_to_region():
    bio = mne.channels.make_standard_montage("biosemi128")
    std = mne.channels.make_standard_montage("standard_1020")

    bio_pos = bio.get_positions()["ch_pos"]
    std_pos = std.get_positions()["ch_pos"]

    target_region_ids = DEFAULT_19_REGION_IDS

    target_std_keys = []
    for target_name in TARGET_19_CANONICAL:
        found = None
        for k in std_pos.keys():
            if _normalize_name(k) == target_name:
                found = k
                break
        if found is None:
            raise ValueError(f"Could not resolve standard_1020 key for target {target_name}")
        target_std_keys.append(found)

    out = {}
    for bch in bio.ch_names:
        bp = bio_pos[bch]
        best_idx = None
        best_dist = None

        for i, std_key in enumerate(target_std_keys):
            tp = std_pos[std_key]
            dist = float(np.linalg.norm(bp - tp))
            if best_dist is None or dist < best_dist:
                best_dist = dist
                best_idx = i

        out[_normalize_name(bch)] = int(target_region_ids[best_idx])

    return out

BIOSEMI128_NAME_TO_REGION = _build_biosemi128_name_to_region()
BIOSEMI128_DEFAULT_ORDER = mne.channels.make_standard_montage("biosemi128").ch_names
BIOSEMI128_DEFAULT_REGION_IDS = np.array(
    [BIOSEMI128_NAME_TO_REGION[_normalize_name(ch)] for ch in BIOSEMI128_DEFAULT_ORDER],
    dtype=np.int64
)

def infer_region_ids_from_channel_names(ch_names: List[str]) -> np.ndarray:
    region_ids = []

    for ch in ch_names:
        s_raw = _normalize_name(ch)
        s = _canonicalize_1020_name(ch)

        if s_raw in BIOSEMI128_NAME_TO_REGION:
            region_ids.append(BIOSEMI128_NAME_TO_REGION[s_raw])
            continue

        if s in {"T7", "T8", "P7", "P8", "FT7", "FT8", "TP7", "TP8", "P9", "P10"}:
            region_ids.append(3)
            continue

        if s.startswith(("FP", "AF", "F")) and not s.startswith(("FC", "FT")):
            region_ids.append(0)
        elif s.startswith(("FC", "C")):
            region_ids.append(1)
        elif s.startswith(("CP", "P", "PO")):
            region_ids.append(2)
        elif s.startswith(("FT", "T", "TP")):
            region_ids.append(3)
        elif s.startswith(("O", "I", "IZ")):
            region_ids.append(4)
        else:
            region_ids.append(1)

    return np.asarray(region_ids, dtype=np.int64)

def infer_region_ids(d: Dict[str, np.ndarray], n_nodes: int) -> np.ndarray:
    direct_keys = [
        "region_ids", "channel_region_id", "channel_region_ids",
        "electrode_region_id", "electrode_region_ids"
    ]
    for k in direct_keys:
        if k in d:
            try:
                arr = np.asarray(d[k]).astype(np.int64).ravel()
                if len(arr) == n_nodes:
                    arr = np.clip(arr, 0, N_REGIONS - 1)
                    return arr
            except Exception:
                pass

    region_name_keys = ["region_names_per_channel", "channel_region_names", "region_names"]
    for k in region_name_keys:
        if k in d:
            try:
                arr = np.asarray(d[k]).ravel().tolist()
                if len(arr) == n_nodes:
                    out = []
                    for x in arr:
                        s = str(x).strip().lower()
                        if s.startswith("front"):
                            out.append(0)
                        elif s.startswith("cent"):
                            out.append(1)
                        elif s.startswith("pari"):
                            out.append(2)
                        elif s.startswith("temp"):
                            out.append(3)
                        elif s.startswith("occi"):
                            out.append(4)
                        else:
                            out.append(1)
                    return np.asarray(out, dtype=np.int64)
            except Exception:
                pass

    ch_names = extract_channel_names_from_npz(d)
    if ch_names is not None and len(ch_names) == n_nodes:
        return infer_region_ids_from_channel_names(ch_names)

    if n_nodes == 128:
        return BIOSEMI128_DEFAULT_REGION_IDS.copy()

    if n_nodes == 19:
        return DEFAULT_19_REGION_IDS.copy()

    out = np.floor(np.linspace(0, N_REGIONS - 1e-6, n_nodes)).astype(np.int64)
    return np.clip(out, 0, N_REGIONS - 1)

# ============================================================
# Region graph + edge region-pair id builders
# ============================================================
def compute_region_adjacency_np(A_single: np.ndarray, region_ids: np.ndarray, n_regions: int = N_REGIONS) -> np.ndarray:
    R = np.zeros((n_regions, n_regions), dtype=np.float32)
    idx_by_region = [np.where(region_ids == r)[0] for r in range(n_regions)]

    for p in range(n_regions):
        Ip = idx_by_region[p]
        for q in range(n_regions):
            Iq = idx_by_region[q]

            if Ip.size == 0 or Iq.size == 0:
                R[p, q] = 0.0
                continue

            block = A_single[np.ix_(Ip, Iq)]

            if p == q:
                if block.shape[0] <= 1:
                    vals = np.array([], dtype=np.float32)
                else:
                    mask = ~np.eye(block.shape[0], dtype=bool)
                    vals = block[mask]
            else:
                vals = block.ravel()

            R[p, q] = float(np.mean(vals)) if vals.size > 0 else 0.0

    R = 0.5 * (R + R.T)
    R = np.clip(R, 0.0, 1.0)
    return R.astype(np.float32)

def build_region_graphs_from_A_np(A_ep: np.ndarray, region_ids: np.ndarray) -> np.ndarray:
    T = A_ep.shape[0]
    R_ep = np.zeros((T, N_REGIONS, N_REGIONS), dtype=np.float32)
    for t in range(T):
        R_ep[t] = compute_region_adjacency_np(A_ep[t], region_ids)
    return R_ep

PAIR_TO_ID = {(p, q): idx for idx, (p, q) in enumerate(
    [(p, q) for p in range(N_REGIONS) for q in range(p, N_REGIONS)]
)}

def build_edge_region_pair_id(region_ids: np.ndarray) -> np.ndarray:
    N = len(region_ids)
    edge_i, edge_j = np.triu_indices(N, k=1)

    out = np.zeros((edge_i.shape[0],), dtype=np.int32)
    for k, (i, j) in enumerate(zip(edge_i, edge_j)):
        p = int(region_ids[i])
        q = int(region_ids[j])
        if p > q:
            p, q = q, p
        out[k] = PAIR_TO_ID[(p, q)]
    return out

def get_or_build_edge_region_pair_id(d: Dict[str, np.ndarray], n_nodes: int) -> np.ndarray:
    if "edge_region_pair_id" in d:
        try:
            arr = np.asarray(d["edge_region_pair_id"]).astype(np.int32).ravel()
            expected_E = n_nodes * (n_nodes - 1) // 2
            if len(arr) == expected_E:
                return arr
        except Exception:
            pass

    region_ids = infer_region_ids(d, n_nodes)
    return build_edge_region_pair_id(region_ids)

def get_or_build_R_all(d: Dict[str, np.ndarray], A_all: np.ndarray) -> np.ndarray:
    if "R_region" in d:
        try:
            R_all = np.asarray(d["R_region"]).astype(np.float32)
            return preprocess_R_np(R_all)
        except Exception:
            pass

    n_nodes = int(A_all.shape[-1])
    region_ids = infer_region_ids(d, n_nodes)

    R_all = np.zeros((A_all.shape[0], A_all.shape[1], N_REGIONS, N_REGIONS), dtype=np.float32)
    for e in range(A_all.shape[0]):
        R_all[e] = build_region_graphs_from_A_np(A_all[e], region_ids)

    return preprocess_R_np(R_all)

# ============================================================
# FAST CACHE PREPARATION
# ============================================================
def prepare_file_cache(dementia_dir: Path, pattern: str):
    """
    Build all missing metadata ONCE per file and cache it.
    This avoids the repeated INFO prints and speeds up iteration.
    """
    files = sorted(dementia_dir.glob(pattern))
    if len(files) == 0:
        raise RuntimeError(f"No files found in {dementia_dir} with pattern {pattern}")

    file_cache = {}
    samples = []

    N = None
    E = None
    vocab = None

    for f in files:
        d = _load_npz_keys(f)
        A_key = _get_A_key(d)

        A_all = preprocess_A_np(np.asarray(d[A_key]).astype(np.float32))
        n_nodes = int(A_all.shape[-1])

        edge_rpid = get_or_build_edge_region_pair_id(d, n_nodes)
        R_all = get_or_build_R_all(d, A_all)

        n_epochs = int(A_all.shape[0])

        if N is None:
            N = n_nodes
            E = int(edge_rpid.shape[0])
            vocab = int(edge_rpid.max()) + 1 if edge_rpid.size > 0 else 1
        else:
            if n_nodes != N:
                raise ValueError(f"N mismatch across files: {f.name} has N={n_nodes}, expected {N}")
            if int(edge_rpid.shape[0]) != E:
                raise ValueError(f"E mismatch across files: {f.name} has E={edge_rpid.shape[0]}, expected {E}")

        file_cache[str(f)] = {
            "A_all": A_all,
            "R_all": R_all,
            "edge_rpid": edge_rpid.astype(np.int32),
        }

        for e in range(n_epochs):
            samples.append((str(f), e, 1))

    return file_cache, samples, N, E, vocab

# ============================================================
# tf.data dataset from cache
# ============================================================
def make_epoch_dataset_from_cache(file_cache, samples, N: int, E: int, shuffle: bool = True):
    def gen():
        local_samples = list(samples)
        if shuffle:
            np.random.shuffle(local_samples)

        for path_str, epoch_idx, label in local_samples:
            cached = file_cache[path_str]
            A = cached["A_all"][epoch_idx].astype(np.float32)
            R = cached["R_all"][epoch_idx].astype(np.float32)
            edge_rpid = cached["edge_rpid"]

            T = A.shape[0]
            peak_mask = np.ones((T,), dtype=np.bool_)

            inputs = {
                "A": A,
                "R": R,
                "peak_mask": peak_mask,
                "edge_rpid": edge_rpid,
            }
            y = np.float32(label)
            yield inputs, y

    output_signature = (
        {
            "A": tf.TensorSpec(shape=(None, N, N), dtype=tf.float32),
            "R": tf.TensorSpec(shape=(None, N_REGIONS, N_REGIONS), dtype=tf.float32),
            "peak_mask": tf.TensorSpec(shape=(None,), dtype=tf.bool),
            "edge_rpid": tf.TensorSpec(shape=(E,), dtype=tf.int32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(samples)), reshuffle_each_iteration=True)

    return ds

# ============================================================
# Stats helpers
# ============================================================
def _quantile_approx(x: tf.Tensor, q: float) -> tf.Tensor:
    x_sorted = tf.sort(x, axis=-1)
    n = tf.shape(x_sorted)[-1]
    idx = tf.cast(
        tf.round(tf.cast(n - 1, tf.float32) * tf.constant(q, tf.float32)),
        tf.int32
    )
    return tf.gather(x_sorted, idx, axis=-1)

def edge_stats(x_edges: tf.Tensor) -> tf.Tensor:
    mean = tf.reduce_mean(x_edges, axis=-1)
    std  = tf.math.reduce_std(x_edges, axis=-1)
    med  = _quantile_approx(x_edges, 0.50)
    q25  = _quantile_approx(x_edges, 0.25)
    q75  = _quantile_approx(x_edges, 0.75)
    mx   = tf.reduce_max(x_edges, axis=-1)
    return tf.stack([mean, std, med, q25, q75, mx], axis=-1)

# ============================================================
# Gather / scatter helpers
# ============================================================
def batch_gather_edges(A: tf.Tensor, edge_i: tf.Tensor, edge_j: tf.Tensor) -> tf.Tensor:
    B = tf.shape(A)[0]
    E = tf.shape(edge_i)[0]

    b_idx = tf.reshape(tf.range(B, dtype=tf.int32), (B, 1, 1))
    b_idx = tf.tile(b_idx, (1, E, 1))

    i = tf.reshape(edge_i, (1, E, 1))
    j = tf.reshape(edge_j, (1, E, 1))
    i = tf.tile(i, (B, 1, 1))
    j = tf.tile(j, (B, 1, 1))

    idx = tf.concat([b_idx, i, j], axis=-1)
    return tf.gather_nd(A, idx)

def scatter_symmetric_edges(edge_i: tf.Tensor, edge_j: tf.Tensor, H_edges: tf.Tensor, N: int) -> tf.Tensor:
    Bsz = tf.shape(H_edges)[0]
    E = tf.shape(edge_i)[0]

    b_idx = tf.reshape(tf.range(Bsz, dtype=tf.int32), (Bsz, 1, 1))
    b_idx = tf.tile(b_idx, (1, E, 1))

    i = tf.reshape(edge_i, (1, E, 1)); i = tf.tile(i, (Bsz, 1, 1))
    j = tf.reshape(edge_j, (1, E, 1)); j = tf.tile(j, (Bsz, 1, 1))

    idx_ij = tf.concat([b_idx, i, j], axis=-1)
    idx_ji = tf.concat([b_idx, j, i], axis=-1)

    idx_all = tf.concat(
        [tf.reshape(idx_ij, (-1, 3)), tf.reshape(idx_ji, (-1, 3))],
        axis=0
    )

    upd = tf.reshape(H_edges, (-1,))
    upd_all = tf.concat([upd, upd], axis=0)

    H0 = tf.zeros((Bsz, N, N), dtype=H_edges.dtype)
    H = tf.tensor_scatter_nd_update(H0, idx_all, upd_all)
    return H

# ============================================================
# Region-pair lookup
# ============================================================
def build_region_pair_lookup_contiguous(n_regions: int = 5):
    ii, jj = [], []
    for p in range(n_regions):
        for q in range(p, n_regions):
            ii.append(p)
            jj.append(q)
    return np.array(ii, dtype=np.int32), np.array(jj, dtype=np.int32)

# ============================================================
# Step 5.1 ThresholdNet
# ============================================================
class ThresholdNet(layers.Layer):
    def __init__(self, peak_emb_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
        ])
        self.out_theta = layers.Dense(1)
        self.out_tau   = layers.Dense(1)

    def call(self, stats_a, stats_r, peak_emb):
        x = tf.concat([stats_a, stats_r, peak_emb], axis=-1)
        h = self.mlp(x)
        theta = self.out_theta(h)
        tau = tf.nn.softplus(self.out_tau(h)) + 1e-6
        return theta, tau

# ============================================================
# Graph-memory gating network
# ============================================================
class GraphGateNet(layers.Layer):
    def __init__(self, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
            layers.Dense(4, activation=None),
        ])

    def call(self, x_graph):
        z = self.mlp(x_graph)
        I_g = tf.sigmoid(z[..., 0])
        F_g = tf.sigmoid(z[..., 1])
        O_g = tf.sigmoid(z[..., 2])
        S_g = tf.sigmoid(z[..., 3])
        return I_g, F_g, O_g, S_g

# ============================================================
# Edge-memory refinement network
# ============================================================
class EdgeRefineNet(layers.Layer):
    def __init__(self, edge_feat_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.mlp = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(hidden, activation="relu"),
            layers.Dense(4, activation=None)
        ])

    def call(self, x_edge):
        z = self.mlp(x_edge)
        I_e = tf.sigmoid(z[..., 0])
        F_e = tf.sigmoid(z[..., 1])
        O_e = tf.sigmoid(z[..., 2])
        Ghat_e = tf.sigmoid(z[..., 3])
        return I_e, F_e, O_e, Ghat_e

# ============================================================
# Step 5.3 Shared GNN
# ============================================================
class SimpleGCN(layers.Layer):
    def __init__(
        self,
        n_nodes: int,
        node_emb_dim: int = 64,
        hidden_dim: int = 64,
        out_dim: int = 128,
        num_layers: int = 3,
        dropout_rate: float = 0.3,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_nodes = n_nodes
        self.node_emb = layers.Embedding(input_dim=n_nodes, output_dim=node_emb_dim)

        self.input_proj = None
        if node_emb_dim != hidden_dim:
            self.input_proj = layers.Dense(hidden_dim, activation="relu")

        self.gcn_layers = [layers.Dense(hidden_dim, activation="relu") for _ in range(num_layers)]
        self.final_layer = layers.Dense(out_dim, activation="relu")

        self.dropout = layers.Dropout(dropout_rate)
        self.node_idx = tf.constant(np.arange(n_nodes, dtype=np.int32))

    def call(self, A, training=False):
        B = tf.shape(A)[0]
        N = self.n_nodes

        X = self.node_emb(self.node_idx)
        X = tf.tile(X[None, :, :], [B, 1, 1])

        if self.input_proj is not None:
            X = self.input_proj(X)

        I = tf.eye(N, batch_shape=[B], dtype=A.dtype)
        A_hat = A + I

        D = tf.reduce_sum(A_hat, axis=-1)
        D_inv_sqrt = tf.pow(tf.maximum(D, 1e-6), -0.5)
        A_norm = (D_inv_sqrt[..., None] * A_hat) * (D_inv_sqrt[:, None, :])

        for dense in self.gcn_layers:
            H = tf.matmul(A_norm, X)
            H = dense(H)
            H = self.dropout(H, training=training)
            X = X + H

        H = tf.matmul(A_norm, X)
        H = self.final_layer(H)
        H = self.dropout(H, training=training)

        z = tf.reduce_mean(H, axis=1)
        return z

# ============================================================
# Step 5.4 Peak attention pooling
# ============================================================
class PeakAttentionPool(layers.Layer):
    def __init__(self, z_dim: int, peak_emb_dim: int, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.score = keras.Sequential([
            layers.Dense(hidden, activation="relu"),
            layers.Dense(1, activation=None),
        ])
        self.z_dim = z_dim
        self.peak_emb_dim = peak_emb_dim

    def call(self, Z, peak_emb_seq, peak_mask):
        x = tf.concat([Z, peak_emb_seq], axis=-1)
        s = tf.squeeze(self.score(x), axis=-1)

        neg_inf = tf.constant(-1e9, dtype=s.dtype)
        s_masked = tf.where(peak_mask, s, neg_inf)
        beta = tf.nn.softmax(s_masked, axis=-1)

        z_pool = tf.reduce_sum(beta[..., None] * Z, axis=1)
        return z_pool, beta

# ============================================================
# Full graph-memory gating + edge-memory refinement model
# ============================================================
class GraphMemoryRefineNetTF(keras.Model):
    def __init__(
        self,
        n_nodes: int,
        region_pair_vocab: int,
        max_peaks: int = 64,
        n_edges: Optional[int] = None,
        d_model: Optional[int] = None,
        r_emb_dim: Optional[int] = None,
        edge_emb_dim: Optional[int] = None,
        summary_dim: Optional[int] = None,
        num_heads: Optional[int] = None,
        ff_dim: Optional[int] = None,
        dropout: Optional[float] = None,
        l2_reg: Optional[float] = None,
        region_pair_emb_dim: int = 8,
        peak_emb_dim: int = 8,
        thr_hidden: int = 64,
        graph_gate_hidden: int = 64,
        edge_gate_hidden: int = 64,
        gnn_out_dim: int = 128,
        gnn_num_layers: int = 3,
        gnn_dropout: float = 0.3,
        clf_hidden1: int = 128,
        clf_hidden2: int = 64,
        clf_dropout: float = 0.2,
        region_presence_theta: float = 0.15,
        region_subset_thr: float = 0.85,
        region_sim_thr: float = 0.60,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_nodes = int(n_nodes)
        self.max_peaks = int(max_peaks)
        self.region_presence_theta = float(region_presence_theta)
        self.region_subset_thr = float(region_subset_thr)
        self.region_sim_thr = float(region_sim_thr)

        edge_i, edge_j = np.triu_indices(self.n_nodes, k=1)
        self.edge_i = tf.constant(edge_i.astype(np.int32))
        self.edge_j = tf.constant(edge_j.astype(np.int32))
        self.E = int(edge_i.shape[0])

        if n_edges is not None and int(n_edges) != self.E:
            raise ValueError(
                f"n_edges={int(n_edges)} does not match upper-triangle edge count "
                f"E={self.E} for n_nodes={self.n_nodes}"
            )

        reg_all_i, reg_all_j = build_region_pair_lookup_contiguous(5)
        self.reg_all_i = tf.constant(reg_all_i, dtype=tf.int32)
        self.reg_all_j = tf.constant(reg_all_j, dtype=tf.int32)

        reg_match_i, reg_match_j = np.triu_indices(5, k=1)
        self.reg_match_i = tf.constant(reg_match_i.astype(np.int32))
        self.reg_match_j = tf.constant(reg_match_j.astype(np.int32))

        self.peak_emb = layers.Embedding(input_dim=self.max_peaks, output_dim=peak_emb_dim)
        self.region_pair_emb = layers.Embedding(input_dim=int(region_pair_vocab), output_dim=region_pair_emb_dim)

        self.threshold_net = ThresholdNet(peak_emb_dim=peak_emb_dim, hidden=thr_hidden)
        self.graph_gate = GraphGateNet(hidden=graph_gate_hidden)

        edge_feat_dim = 9 + region_pair_emb_dim + peak_emb_dim
        self.edge_refine = EdgeRefineNet(edge_feat_dim=edge_feat_dim, hidden=edge_gate_hidden)

        self.gnn = SimpleGCN(
            n_nodes=self.n_nodes,
            node_emb_dim=64,
            hidden_dim=64,
            out_dim=gnn_out_dim,
            num_layers=gnn_num_layers,
            dropout_rate=gnn_dropout
        )

        self.peak_pool = PeakAttentionPool(z_dim=gnn_out_dim, peak_emb_dim=peak_emb_dim)

        self.classifier = keras.Sequential([
            layers.Dense(clf_hidden1, activation="relu"),
            layers.Dropout(clf_dropout),
            layers.Dense(clf_hidden2, activation="relu"),
            layers.Dropout(clf_dropout),
            layers.Dense(1, activation="sigmoid")
        ])

        self.last_beta = None
        self.last_theta = None
        self.last_tau = None
        self.last_graph_subset = None
        self.last_graph_jaccard = None
        self.last_graph_update = None

    def _region_match_gate(self, C_graph_prev, R_k, mask_b):
        prev_pairs = batch_gather_edges(C_graph_prev, self.reg_match_i, self.reg_match_j)
        curr_pairs = batch_gather_edges(R_k,          self.reg_match_i, self.reg_match_j)

        prev_active_b = prev_pairs >= self.region_presence_theta
        curr_active_b = curr_pairs >= self.region_presence_theta

        inter = tf.reduce_sum(tf.cast(tf.logical_and(prev_active_b, curr_active_b), tf.float32), axis=-1)
        prev_count = tf.reduce_sum(tf.cast(prev_active_b, tf.float32), axis=-1)
        union = tf.reduce_sum(tf.cast(tf.logical_or(prev_active_b, curr_active_b), tf.float32), axis=-1)

        subset_score = tf.where(prev_count > 0.0, inter / (prev_count + 1e-6), tf.zeros_like(inter))
        jaccard = tf.where(union > 0.0, inter / (union + 1e-6), tf.zeros_like(inter))

        subset_gate = tf.cast(subset_score >= self.region_subset_thr, tf.float32)
        sim_gate = tf.cast(jaccard >= self.region_sim_thr, tf.float32)
        update_gate = tf.maximum(subset_gate, sim_gate)

        update_gate = mask_b * update_gate
        subset_gate = mask_b * subset_gate
        subset_score = mask_b * subset_score
        jaccard = mask_b * jaccard

        return update_gate, subset_score, jaccard, subset_gate

    def call(self, inputs, training=False):
        A = inputs["A"]
        R = inputs["R"]
        peak_mask = inputs["peak_mask"]
        edge_rpid = inputs["edge_rpid"]

        B = tf.shape(A)[0]
        T = tf.shape(A)[1]
        N = self.n_nodes

        if edge_rpid.shape.rank == 2:
            edge_rpid_1d = tf.cast(edge_rpid[0], tf.int32)
        else:
            edge_rpid_1d = tf.cast(edge_rpid, tf.int32)

        rp_emb = self.region_pair_emb(edge_rpid_1d)
        edge_reg_i = tf.gather(self.reg_all_i, edge_rpid_1d)
        edge_reg_j = tf.gather(self.reg_all_j, edge_rpid_1d)

        peak_ids = tf.range(T, dtype=tf.int32)
        peak_ids = tf.minimum(peak_ids, tf.constant(self.max_peaks - 1, tf.int32))
        peak_emb_seq = self.peak_emb(peak_ids)
        peak_emb_seq_B = tf.tile(peak_emb_seq[None, :, :], [B, 1, 1])

        C_graph = tf.zeros((B, 5, 5), dtype=tf.float32)
        C_edge  = tf.zeros((B, self.E), dtype=tf.float32)

        Z_list = tf.TensorArray(tf.float32, size=T)
        theta_list = tf.TensorArray(tf.float32, size=T)
        tau_list = tf.TensorArray(tf.float32, size=T)
        subset_list = tf.TensorArray(tf.float32, size=T)
        jaccard_list = tf.TensorArray(tf.float32, size=T)
        update_list = tf.TensorArray(tf.float32, size=T)

        def loop_body(k, C_graph, C_edge, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list):
            A_k = A[:, k, :, :]
            R_k = R[:, k, :, :]

            mask_b = tf.cast(peak_mask[:, k], tf.float32)
            mask_edge = mask_b[:, None]

            R_k = 0.5 * (R_k + tf.transpose(R_k, perm=[0, 2, 1]))
            R_k = tf.clip_by_value(R_k, 0.0, 1.0)
            R_k = mask_b[:, None, None] * R_k

            A_edges = batch_gather_edges(A_k, self.edge_i, self.edge_j)
            stats_a = edge_stats(A_edges)

            R_edges = batch_gather_edges(R_k, self.reg_match_i, self.reg_match_j)
            stats_r = edge_stats(R_edges)

            pk_emb = peak_emb_seq_B[:, k, :]

            theta_k, tau_k = self.threshold_net(stats_a, stats_r, pk_emb)
            theta_k = tf.squeeze(theta_k, axis=-1)
            tau_k   = tf.squeeze(tau_k, axis=-1)

            theta_k = mask_b * theta_k
            tau_k   = mask_b * tau_k + (1.0 - mask_b) * 1.0

            theta_mat = theta_k[:, None, None]
            tau_mat   = tau_k[:, None, None]

            M_k = tf.sigmoid((A_k - theta_mat) / tau_mat)
            M_k = mask_b[:, None, None] * M_k
            Abar_k = M_k * A_k
            Abar_k = tf.linalg.set_diag(Abar_k, tf.zeros((B, N), dtype=Abar_k.dtype))

            def first_peak_branch():
                ones = mask_b
                C_graph_new = R_k
                H_graph = R_k
                return ones, ones, ones, ones, C_graph_new, H_graph

            def later_peak_branch():
                graph_update_gate, subset_score, jaccard_score, subset_gate = self._region_match_gate(C_graph, R_k, mask_b)

                prev_pairs = batch_gather_edges(C_graph, self.reg_match_i, self.reg_match_j)
                curr_pairs = batch_gather_edges(R_k,     self.reg_match_i, self.reg_match_j)
                stats_prev = edge_stats(prev_pairs)
                stats_curr = edge_stats(curr_pairs)
                stats_diff = tf.abs(stats_curr - stats_prev)

                x_graph = tf.concat(
                    [
                        stats_prev,
                        stats_curr,
                        stats_diff,
                        subset_score[:, None],
                        jaccard_score[:, None],
                        pk_emb
                    ],
                    axis=-1
                )

                I_g, F_g, O_g, S_g = self.graph_gate(x_graph)

                R_subset = tf.maximum(C_graph, R_k)
                R_sim = 0.5 * (C_graph + R_k)

                Sg_mat = S_g[:, None, None]
                I_mat  = I_g[:, None, None]
                F_mat  = F_g[:, None, None]
                O_mat  = O_g[:, None, None]
                Ug_mat = graph_update_gate[:, None, None]

                graph_cand = Sg_mat * R_subset + (1.0 - Sg_mat) * R_sim
                C_graph_prop = F_mat * C_graph + I_mat * graph_cand
                C_graph_prop = tf.clip_by_value(C_graph_prop, 0.0, 1.0)

                C_graph_new = Ug_mat * C_graph_prop + (1.0 - Ug_mat) * C_graph
                C_graph_new = tf.clip_by_value(C_graph_new, 0.0, 1.0)

                H_graph = O_mat * C_graph_new
                H_graph = tf.clip_by_value(H_graph, 0.0, 1.0)

                return graph_update_gate, subset_score, jaccard_score, subset_gate, C_graph_new, H_graph

            graph_update_gate, subset_score, jaccard_score, subset_gate, C_graph_new, H_graph = tf.cond(
                tf.equal(k, 0),
                first_peak_branch,
                later_peak_branch
            )

            Abar_edges = batch_gather_edges(Abar_k, self.edge_i, self.edge_j)
            diff_prev = tf.abs(Abar_edges - C_edge)

            graph_prior_edge = batch_gather_edges(H_graph, edge_reg_i, edge_reg_j)
            diff_mem_graph = tf.abs(graph_prior_edge - C_edge)
            diff_input_graph = tf.abs(Abar_edges - graph_prior_edge)

            rp = tf.tile(rp_emb[None, :, :], [B, 1, 1])
            pk = tf.tile(pk_emb[:, None, :], [1, self.E, 1])

            graph_update_e = tf.tile(graph_update_gate[:, None, None], [1, self.E, 1])
            subset_e       = tf.tile(subset_score[:, None, None],      [1, self.E, 1])
            jaccard_e      = tf.tile(jaccard_score[:, None, None],     [1, self.E, 1])

            x_edge = tf.concat([
                Abar_edges[..., None],
                C_edge[..., None],
                diff_prev[..., None],
                graph_prior_edge[..., None],
                diff_mem_graph[..., None],
                diff_input_graph[..., None],
                graph_update_e,
                subset_e,
                jaccard_e,
                rp,
                pk
            ], axis=-1)

            I_e, F_e, O_e, Ghat_e = self.edge_refine(x_edge)

            C_edge_prop = F_e * C_edge + I_e * Ghat_e
            C_edge_prop = tf.clip_by_value(C_edge_prop, 0.0, 1.0)

            refine_alpha = graph_update_gate[:, None] * tf.clip_by_value(
                0.5 * subset_score[:, None] + 0.5 * jaccard_score[:, None],
                0.0, 1.0
            )

            C_edge_refined = refine_alpha * graph_prior_edge + (1.0 - refine_alpha) * C_edge_prop
            C_edge_refined = tf.clip_by_value(C_edge_refined, 0.0, 1.0)

            C_edge_new = graph_update_gate[:, None] * C_edge_refined + (1.0 - graph_update_gate[:, None]) * C_edge
            C_edge_new = tf.clip_by_value(C_edge_new, 0.0, 1.0)

            H_prev = C_edge
            H_prop = O_e * C_edge_new
            H_edges = graph_update_gate[:, None] * H_prop + (1.0 - graph_update_gate[:, None]) * H_prev
            H_edges = tf.clip_by_value(H_edges, 0.0, 1.0)

            C_edge_new = mask_edge * C_edge_new + (1.0 - mask_edge) * C_edge
            H_edges    = mask_edge * H_edges    + (1.0 - mask_edge) * H_prev
            C_graph_new = mask_b[:, None, None] * C_graph_new + (1.0 - mask_b[:, None, None]) * C_graph

            H_mat = scatter_symmetric_edges(self.edge_i, self.edge_j, H_edges, N)
            H_mat = tf.clip_by_value(H_mat, 0.0, 1.0)

            z_k = self.gnn(H_mat, training=training)
            z_k = z_k * mask_b[:, None]

            Z_list = Z_list.write(k, z_k)
            theta_list = theta_list.write(k, theta_k)
            tau_list = tau_list.write(k, tau_k)
            subset_list = subset_list.write(k, subset_score)
            jaccard_list = jaccard_list.write(k, jaccard_score)
            update_list = update_list.write(k, graph_update_gate)

            return (
                k + 1,
                C_graph_new,
                C_edge_new,
                Z_list,
                theta_list,
                tau_list,
                subset_list,
                jaccard_list,
                update_list
            )

        def loop_cond(k, C_graph, C_edge, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list):
            return k < T

        k0 = tf.constant(0, dtype=tf.int32)
        _, _, _, Z_list, theta_list, tau_list, subset_list, jaccard_list, update_list = tf.while_loop(
            loop_cond,
            loop_body,
            loop_vars=[
                k0,
                C_graph,
                C_edge,
                Z_list,
                theta_list,
                tau_list,
                subset_list,
                jaccard_list,
                update_list
            ],
            parallel_iterations=1
        )

        Z = tf.transpose(Z_list.stack(), perm=[1, 0, 2])

        z_pool, beta = self.peak_pool(Z, peak_emb_seq_B, peak_mask)
        probs = tf.squeeze(self.classifier(z_pool, training=training), axis=-1)

        self.last_beta = beta
        self.last_theta = tf.transpose(theta_list.stack(), perm=[1, 0])
        self.last_tau   = tf.transpose(tau_list.stack(),   perm=[1, 0])
        self.last_graph_subset  = tf.transpose(subset_list.stack(),  perm=[1, 0])
        self.last_graph_jaccard = tf.transpose(jaccard_list.stack(), perm=[1, 0])
        self.last_graph_update  = tf.transpose(update_list.stack(),  perm=[1, 0])

        return probs

# keep old name for compatibility
GraphMemoryNetTF = GraphMemoryRefineNetTF

# ============================================================
# FAST SANITY CHECK
# ============================================================
file_cache, samples, N, E, vocab = prepare_file_cache(DEMENTIA_DIR, FILE_PATTERN)

print("Total dementia epoch samples:", len(samples))
print("N nodes:", N, "E edges:", E, "region_pair_vocab:", vocab)

ds = make_epoch_dataset_from_cache(file_cache, samples, N=N, E=E, shuffle=True)

BATCH_SIZE = 8
ds = ds.padded_batch(
    BATCH_SIZE,
    padded_shapes=(
        {
            "A": (None, N, N),
            "R": (None, N_REGIONS, N_REGIONS),
            "peak_mask": (None,),
            "edge_rpid": (E,),
        },
        ()
    ),
    padding_values=(
        {
            "A": tf.constant(0.0, tf.float32),
            "R": tf.constant(0.0, tf.float32),
            "peak_mask": tf.constant(False, tf.bool),
            "edge_rpid": tf.constant(0, tf.int32),
        },
        tf.constant(1.0, tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

model = GraphMemoryRefineNetTF(
    n_nodes=N,
    region_pair_vocab=vocab,
    max_peaks=64,
    region_pair_emb_dim=8,
    peak_emb_dim=8,
    thr_hidden=64,
    graph_gate_hidden=64,
    edge_gate_hidden=64,
    gnn_out_dim=128,
    gnn_num_layers=3,
    gnn_dropout=0.3,
    clf_hidden1=128,
    clf_hidden2=64,
    clf_dropout=0.2,
    region_presence_theta=0.15,
    region_subset_thr=0.85,
    region_sim_thr=0.60,
)

batch_inputs, batch_y = next(iter(ds))
probs = model(batch_inputs, training=False)

print("output shape:", probs.shape)
print("beta shape:", model.last_beta.shape if model.last_beta is not None else None)
print("theta shape:", model.last_theta.shape if model.last_theta is not None else None)
print("tau shape:", model.last_tau.shape if model.last_tau is not None else None)
print("graph subset shape:", model.last_graph_subset.shape if model.last_graph_subset is not None else None)
print("graph jaccard shape:", model.last_graph_jaccard.shape if model.last_graph_jaccard is not None else None)
print("graph update shape:", model.last_graph_update.shape if model.last_graph_update is not None else None)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.5)]
)

print("\nModel compiled successfully.")


In [ ]:
# ============================================================
# HiGMR-Net classification code
# Subject-wise Stratified 5-Fold CV
#
# ============================================================

import numpy as np
from pathlib import Path
import tensorflow as tf
import time
from datetime import timedelta

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

# ----------------------------
# REQUIREMENT: Step 5 model exists
# ----------------------------
if "GraphMemoryRefineNetTF" in globals():
    HiGMRNet = GraphMemoryRefineNetTF
elif "GraphMemoryNetTF" in globals():
    HiGMRNet = GraphMemoryNetTF
else:
    raise NameError("HiGMR-Net model is not defined. Run Step 5 first.")

# ----------------------------
# Disable XLA/JIT
# ----------------------------
try:
    tf.config.optimizer.set_jit(False)
except Exception:
    pass

# ----------------------------
# PATHS
# ----------------------------
HEALTHY_DIR = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity/graphs_subjectwise/Healthy") # change the paths
DEMENTIA_DIR = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity/graphs_subjectwise/Dementia")
FILE_PATTERN = "*_with_regions.npz"

OUT_DIR = Path("/content/drive/My Drive/XXX/Temp_Gated_Connectivity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# SETTINGS
# ----------------------------
K_FOLDS = 5
SEED = 42

VAL_FRAC_PER_CLASS = 0.30
MIN_VAL_SUBJECTS_PER_CLASS = 2

BATCH_SIZE = 8
EPOCHS = 15
LR = 2e-4

WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.0

EDGE_DROPOUT = 0.0
A_NOISE_STD = 0.0

CHECKPOINT_OBJECTIVE = "acc"       # "auc", "acc", "bal_acc"
THRESHOLD_OBJECTIVE = "bal_acc"    # "acc" or "bal_acc"
EARLY_STOP_PATIENCE = 8

USE_CLASS_WEIGHTS = False

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# ============================================================
# Time utilities
# ============================================================
def format_time(seconds):
    """Return readable time string."""
    return str(timedelta(seconds=int(round(seconds))))


class EpochTimeLogger(tf.keras.callbacks.Callback):
    """
    Prints epoch duration and cumulative model.fit time.
    Note: time includes validation and callback operations.
    """
    def on_train_begin(self, logs=None):
        self.train_start_time = time.perf_counter()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.perf_counter() - self.epoch_start_time
        total_time = time.perf_counter() - self.train_start_time
        print(
            f"[Time] Epoch {epoch + 1:02d} duration: "
            f"{format_time(epoch_time)} ({epoch_time:.2f} sec) | "
            f"Cumulative training time: {format_time(total_time)} ({total_time:.2f} sec)"
        )

# ============================================================
# Utils
# ============================================================
def list_files(folder: Path, pattern: str):
    files = sorted(folder.glob(pattern))
    if len(files) == 0:
        raise RuntimeError(f"No files found in {folder} with pattern {pattern}")
    return files


def _load_npz_keys(npz_path: Path):
    with np.load(npz_path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}


def _get_A_key(d):
    if "A_pcorr" in d:
        return "A_pcorr"
    if "A" in d:
        return "A"
    raise KeyError("Missing 'A_pcorr' or 'A' in npz.")


def infer_constants_and_max_peaks(files):
    N = None
    E = None
    vocab = None
    max_peaks = 0

    for fp in files:
        d = _load_npz_keys(Path(fp))
        A_key = _get_A_key(d)
        A_all = d[A_key]

        if N is None:
            N = int(A_all.shape[-1])
            edge_rpid = d["edge_region_pair_id"].astype(np.int64)
            E = int(edge_rpid.shape[0])
            vocab = int(edge_rpid.max()) + 1

        max_peaks = max(max_peaks, int(A_all.shape[1]))

    return N, E, vocab, max_peaks


def pick_val_subjects(train_paths, train_labels, rng, val_frac=0.30, min_per_class=2):
    train_paths = np.array(train_paths, dtype=object)
    train_labels = np.array(train_labels, dtype=int)

    hc_idx = np.where(train_labels == 0)[0]
    dem_idx = np.where(train_labels == 1)[0]

    n_hc_val = max(min_per_class, int(round(len(hc_idx) * val_frac)))
    n_dem_val = max(min_per_class, int(round(len(dem_idx) * val_frac)))

    n_hc_val = min(n_hc_val, len(hc_idx) - 1)
    n_dem_val = min(n_dem_val, len(dem_idx) - 1)

    if n_hc_val < 1 or n_dem_val < 1:
        raise ValueError(
            f"Not enough subjects to create validation split safely. "
            f"HC={len(hc_idx)}, Dem={len(dem_idx)}"
        )

    val_hc = rng.choice(hc_idx, size=n_hc_val, replace=False)
    val_dem = rng.choice(dem_idx, size=n_dem_val, replace=False)
    val_idx = np.concatenate([val_hc, val_dem])

    val_idx_set = set(val_idx.tolist())
    keep_idx = np.array([i for i in range(len(train_paths)) if i not in val_idx_set], dtype=int)

    val_paths = train_paths[val_idx]
    val_labels = train_labels[val_idx]
    train_paths_final = train_paths[keep_idx]
    train_labels_final = train_labels[keep_idx]

    return train_paths_final, train_labels_final, val_paths, val_labels


def compute_class_weights_from_subjects(subject_labels):
    subject_labels = np.array(subject_labels, dtype=int)
    n0 = int((subject_labels == 0).sum())
    n1 = int((subject_labels == 1).sum())
    total = n0 + n1

    if n0 == 0 or n1 == 0:
        return None

    return {0: float(total / (2.0 * n0)), 1: float(total / (2.0 * n1))}


def build_epoch_samples_from_subjects(subject_paths, subject_labels, class_w=None):
    samples = []
    for p, lab in zip(subject_paths, subject_labels):
        p = str(p)
        lab = int(lab)

        d = _load_npz_keys(Path(p))
        A_key = _get_A_key(d)
        n_epochs = int(d[A_key].shape[0])

        if class_w is None:
            w = 1.0 / max(n_epochs, 1)
        else:
            w = float(class_w[lab]) / max(n_epochs, 1)

        for e in range(n_epochs):
            samples.append((p, int(e), lab, np.float32(w)))

    return samples

# ============================================================
# Graph preprocessing
# ============================================================
def preprocess_A_np(A):
    """
    If correlation matrices contain signed values in [-1, 1], preserve them
    by mapping to [0, 1]. If already nonnegative, keep [0, 1].
    """
    A = np.nan_to_num(A, nan=0.0, posinf=1.0, neginf=-1.0).astype(np.float32)
    A = 0.5 * (A + np.swapaxes(A, -1, -2))

    idx = np.arange(A.shape[-1])
    A[..., idx, idx] = 0.0

    if np.nanmin(A) < 0.0:
        A = np.clip(A, -1.0, 1.0)
        A = 0.5 * (A + 1.0)
    else:
        A = np.clip(A, 0.0, 1.0)

    A[..., idx, idx] = 0.0
    return A

# ============================================================
# tf.data builders
# ============================================================
def make_epoch_dataset(samples, N: int, E: int, MAX_PEAKS: int, shuffle: bool):
    state = {"call_count": 0}

    def gen():
        rng_local = np.random.default_rng(SEED + state["call_count"])
        state["call_count"] += 1

        local_samples = list(samples)
        if shuffle:
            rng_local.shuffle(local_samples)

        cache_path = None
        cache = None

        for path_str, epoch_idx, label, sw in local_samples:
            path_str = str(path_str)
            epoch_idx = int(epoch_idx)
            label = int(label)
            sw = np.float32(sw)

            if cache_path != path_str:
                cache = _load_npz_keys(Path(path_str))
                cache_path = path_str

            d = cache
            A_key = _get_A_key(d)
            A_all = d[A_key].astype(np.float32)
            R_all = np.nan_to_num(d["R_region"].astype(np.float32), nan=0.0)
            edge_rpid = d["edge_region_pair_id"].astype(np.int32)

            A_ep = preprocess_A_np(A_all[epoch_idx])
            R_ep = np.nan_to_num(R_all[epoch_idx], nan=0.0).astype(np.float32)

            T = int(A_ep.shape[0])
            t_use = min(T, MAX_PEAKS)

            A_pad = np.zeros((MAX_PEAKS, N, N), dtype=np.float32)
            R_pad = np.zeros((MAX_PEAKS, 5, 5), dtype=np.float32)
            peak_mask = np.zeros((MAX_PEAKS,), dtype=np.bool_)

            A_pad[:t_use] = A_ep[:t_use]
            R_pad[:t_use] = R_ep[:t_use]
            peak_mask[:t_use] = True

            x = {
                "A": A_pad,
                "R": R_pad,
                "peak_mask": peak_mask,
                "edge_rpid": edge_rpid,
            }
            y = np.float32(label)
            yield x, y, sw

    output_signature = (
        {
            "A": tf.TensorSpec(shape=(MAX_PEAKS, N, N), dtype=tf.float32),
            "R": tf.TensorSpec(shape=(MAX_PEAKS, 5, 5), dtype=tf.float32),
            "peak_mask": tf.TensorSpec(shape=(MAX_PEAKS,), dtype=tf.bool),
            "edge_rpid": tf.TensorSpec(shape=(E,), dtype=tf.int32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32),
    )

    return tf.data.Dataset.from_generator(gen, output_signature=output_signature)


def make_batched(ds, batch_size: int):
    return ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)


def augment_graph_batch(x, y, sw):
    # Disabled for HiGMR-Net to keep A and R consistent
    return x, y, sw

# ============================================================
# Training helpers
# ============================================================
def make_optimizer(lr, weight_decay):
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=lr,
            weight_decay=weight_decay,
            clipnorm=1.0,
        )
    except Exception:
        return tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)


def make_auc_metric(name="auc"):
    return tf.keras.metrics.AUC(name=name, curve="ROC")

# ============================================================
# Subject-level evaluation
# ============================================================
def subject_mean_proba(model, subject_path: str, N: int, E: int, MAX_PEAKS: int, batch_size: int = 32):
    d = _load_npz_keys(Path(subject_path))
    A_key = _get_A_key(d)

    A_all = preprocess_A_np(d[A_key].astype(np.float32))
    R_all = np.nan_to_num(d["R_region"].astype(np.float32), nan=0.0)
    edge_rpid = d["edge_region_pair_id"].astype(np.int32)

    n_epochs = int(A_all.shape[0])
    T = int(A_all.shape[1])
    t_use = min(T, MAX_PEAKS)

    A_pad = np.zeros((n_epochs, MAX_PEAKS, N, N), dtype=np.float32)
    R_pad = np.zeros((n_epochs, MAX_PEAKS, 5, 5), dtype=np.float32)
    pm = np.zeros((n_epochs, MAX_PEAKS), dtype=bool)

    A_pad[:, :t_use] = A_all[:, :t_use]
    R_pad[:, :t_use] = R_all[:, :t_use]
    pm[:, :t_use] = True

    edge_rep = np.repeat(edge_rpid[None, :], n_epochs, axis=0)

    probs = model.predict(
        {"A": A_pad, "R": R_pad, "peak_mask": pm, "edge_rpid": edge_rep},
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    return float(np.mean(probs))


def collect_subject_probs(model, paths, labels, N, E, MAX_PEAKS):
    probs = []
    ys = []
    for p, y in zip(paths, labels):
        probs.append(subject_mean_proba(model, str(p), N, E, MAX_PEAKS))
        ys.append(int(y))
    return np.array(probs, dtype=np.float32), np.array(ys, dtype=np.int32)


def find_best_subject_threshold(probs, labels, objective="bal_acc"):
    thresholds = np.unique(np.r_[np.linspace(0.05, 0.95, 181), probs])

    best_thr = 0.5
    best_score = -1.0

    for thr in thresholds:
        preds = (probs >= thr).astype(np.int32)

        if objective == "acc":
            score = accuracy_score(labels, preds)
        elif objective == "bal_acc":
            score = balanced_accuracy_score(labels, preds)
        else:
            raise ValueError(f"Unknown objective: {objective}")

        if (score > best_score) or (
            np.isclose(score, best_score) and abs(thr - 0.5) < abs(best_thr - 0.5)
        ):
            best_score = float(score)
            best_thr = float(thr)

    return best_thr, best_score


def subject_accuracy_from_probs(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(np.int32)
    return float((preds == labels).mean())

# ============================================================
# Custom callback: choose model by SUBJECT-LEVEL validation metric
# ============================================================
class SubjectValCheckpoint(tf.keras.callbacks.Callback):
    def __init__(
        self,
        val_paths,
        val_labels,
        N,
        E,
        MAX_PEAKS,
        checkpoint_objective="auc",
        threshold_objective="bal_acc",
        patience=8,
        min_delta=1e-4,
    ):
        super().__init__()
        self.val_paths = val_paths
        self.val_labels = val_labels
        self.N = N
        self.E = E
        self.MAX_PEAKS = MAX_PEAKS
        self.checkpoint_objective = checkpoint_objective
        self.threshold_objective = threshold_objective
        self.patience = patience
        self.min_delta = min_delta

        self.best_score = -np.inf
        self.best_acc = -np.inf
        self.best_bal_acc = -np.inf
        self.best_auc = -np.inf
        self.best_thr = 0.5
        self.best_weights = None
        self.history_scores = []
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        val_probs, val_y = collect_subject_probs(
            self.model, self.val_paths, self.val_labels, self.N, self.E, self.MAX_PEAKS
        )

        thr_acc, best_acc = find_best_subject_threshold(val_probs, val_y, objective="acc")
        thr_bal, best_bal = find_best_subject_threshold(val_probs, val_y, objective="bal_acc")
        preds05 = (val_probs >= 0.5).astype(np.int32)

        acc05 = accuracy_score(val_y, preds05)
        bal05 = balanced_accuracy_score(val_y, preds05)

        try:
            auc = roc_auc_score(val_y, val_probs)
        except Exception:
            auc = np.nan

        if self.checkpoint_objective == "auc":
            score = -np.inf if np.isnan(auc) else float(auc)
        elif self.checkpoint_objective == "acc":
            score = float(best_acc)
        elif self.checkpoint_objective == "bal_acc":
            score = float(best_bal)
        else:
            raise ValueError(f"Unknown checkpoint objective: {self.checkpoint_objective}")

        chosen_thr = thr_acc if self.threshold_objective == "acc" else thr_bal

        logs["subject_val_auc"] = float(auc) if not np.isnan(auc) else np.nan
        logs["subject_val_acc05"] = float(acc05)
        logs["subject_val_bal05"] = float(bal05)
        logs["subject_val_best_acc"] = float(best_acc)
        logs["subject_val_best_bal"] = float(best_bal)

        self.history_scores.append((epoch + 1, acc05, bal05, best_acc, best_bal, auc))
        print(
            f"\n[SubjectVal] epoch={epoch+1:02d} "
            f"auc={auc:.4f} "
            f"acc@0.5={acc05*100:.2f}% "
            f"bal@0.5={bal05*100:.2f}% "
            f"best_acc={best_acc*100:.2f}% thr={thr_acc:.3f} "
            f"best_bal={best_bal*100:.2f}% thr={thr_bal:.3f}"
        )

        if score > self.best_score + self.min_delta:
            self.best_score = float(score)
            self.best_acc = float(best_acc)
            self.best_bal_acc = float(best_bal)
            self.best_auc = float(auc) if not np.isnan(auc) else self.best_auc
            self.best_thr = float(chosen_thr)
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"[SubjectVal] early stopping at epoch {epoch+1}")
                self.model.stop_training = True

# ============================================================
# MAIN
# ============================================================
total_run_start_time = time.perf_counter()

healthy_files = list_files(HEALTHY_DIR, FILE_PATTERN)
dementia_files = list_files(DEMENTIA_DIR, FILE_PATTERN)

subjects = [(str(f), 0) for f in healthy_files] + [(str(f), 1) for f in dementia_files]
subject_paths = np.array([s[0] for s in subjects], dtype=object)
subject_labels = np.array([s[1] for s in subjects], dtype=int)

print(f"Total subjects: {len(subject_paths)} (HC={len(healthy_files)}, Dem={len(dementia_files)})")

all_files = list(subject_paths)
N, E, vocab, MAX_PEAKS = infer_constants_and_max_peaks(all_files)
print(f"Inferred constants: N={N}, E={E}, vocab={vocab}, MAX_PEAKS={MAX_PEAKS}")

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_epoch_test_acc = []
fold_subject_val_acc = []
fold_subject_test_acc = []
fold_subject_test_bal_acc = []
fold_subject_test_auc = []
fold_best_thresholds = []

# New timing lists
fold_training_times = []
fold_total_times = []

rng = np.random.default_rng(SEED)

for fold, (train_idx, test_idx) in enumerate(skf.split(subject_paths, subject_labels), start=1):
    fold_start_time = time.perf_counter()

    print(f"\n{'='*70}\nFOLD {fold}/{K_FOLDS}\n{'='*70}")

    train_paths_outer = subject_paths[train_idx]
    train_labels_outer = subject_labels[train_idx]
    test_paths = subject_paths[test_idx]
    test_labels = subject_labels[test_idx]

    train_paths, train_labels, val_paths, val_labels = pick_val_subjects(
        train_paths_outer,
        train_labels_outer,
        rng,
        val_frac=VAL_FRAC_PER_CLASS,
        min_per_class=MIN_VAL_SUBJECTS_PER_CLASS,
    )

    print(f"Subjects (outer train): {len(train_paths_outer)} | (outer test): {len(test_paths)}")
    print(f"Subjects (inner train): {len(train_paths)} | (inner val): {len(val_paths)}")
    print(f"Val subjects: {[Path(p).name for p in val_paths]}")

    class_w = compute_class_weights_from_subjects(train_labels) if USE_CLASS_WEIGHTS else None
    print("Class weights:", class_w)

    train_samples = build_epoch_samples_from_subjects(train_paths, train_labels, class_w=class_w)
    val_samples = build_epoch_samples_from_subjects(val_paths, val_labels, class_w=None)
    test_samples = build_epoch_samples_from_subjects(test_paths, test_labels, class_w=None)

    print(f"Epochs: train={len(train_samples)} | val={len(val_samples)} | test={len(test_samples)}")

    ds_train = make_epoch_dataset(train_samples, N=N, E=E, MAX_PEAKS=MAX_PEAKS, shuffle=True)
    ds_train = make_batched(ds_train, batch_size=BATCH_SIZE)
    ds_train = ds_train.map(augment_graph_batch, num_parallel_calls=tf.data.AUTOTUNE)

    ds_val = make_epoch_dataset(val_samples, N=N, E=E, MAX_PEAKS=MAX_PEAKS, shuffle=False)
    ds_val = make_batched(ds_val, batch_size=BATCH_SIZE)

    ds_test = make_epoch_dataset(test_samples, N=N, E=E, MAX_PEAKS=MAX_PEAKS, shuffle=False)
    ds_test = make_batched(ds_test, batch_size=BATCH_SIZE)

    tf.keras.backend.clear_session()

    model = HiGMRNet(
        n_nodes=N,
        region_pair_vocab=vocab,
        max_peaks=MAX_PEAKS,
        region_pair_emb_dim=8,
        peak_emb_dim=8,
        thr_hidden=64,
        graph_gate_hidden=64,
        edge_gate_hidden=64,
        gnn_out_dim=128,
        gnn_num_layers=3,
        gnn_dropout=0.3,
        clf_hidden1=128,
        clf_hidden2=64,
        clf_dropout=0.3,
        region_presence_theta=0.15,
        region_subset_thr=0.85,
        region_sim_thr=0.60,
    )

    dummy_x = {
        "A": tf.zeros((1, MAX_PEAKS, N, N), dtype=tf.float32),
        "R": tf.zeros((1, MAX_PEAKS, 5, 5), dtype=tf.float32),
        "peak_mask": tf.ones((1, MAX_PEAKS), dtype=tf.bool),
        "edge_rpid": tf.zeros((1, E), dtype=tf.int32),
    }
    _ = model(dummy_x, training=False)

    opt = make_optimizer(LR, WEIGHT_DECAY)

    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.BinaryCrossentropy(
            from_logits=False,
            label_smoothing=LABEL_SMOOTHING,
        ),
        metrics=[],
        weighted_metrics=[
            tf.keras.metrics.BinaryAccuracy(name="acc", threshold=0.5),
            make_auc_metric(name="auc"),
        ],
        jit_compile=False,
    )

    subject_ckpt = SubjectValCheckpoint(
        val_paths=val_paths,
        val_labels=val_labels,
        N=N,
        E=E,
        MAX_PEAKS=MAX_PEAKS,
        checkpoint_objective=CHECKPOINT_OBJECTIVE,
        threshold_objective=THRESHOLD_OBJECTIVE,
        patience=EARLY_STOP_PATIENCE,
    )

    callbacks = [
        subject_ckpt,
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="subject_val_auc" if CHECKPOINT_OBJECTIVE == "auc" else (
                "subject_val_best_acc" if CHECKPOINT_OBJECTIVE == "acc" else "subject_val_best_bal"
            ),
            mode="max",
            factor=0.3,
            patience=8,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.TerminateOnNaN(),
        EpochTimeLogger(),
    ]

    # ----------------------------
    # Training with timing
    # ----------------------------
    print(f"\n[Time] Starting training for Fold {fold}...")
    training_start_time = time.perf_counter()

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=2,
    )

    training_elapsed = time.perf_counter() - training_start_time
    fold_training_times.append(training_elapsed)

    print(
        f"\n[Time] Fold {fold} model.fit training time: "
        f"{format_time(training_elapsed)} ({training_elapsed:.2f} sec)"
    )

    if subject_ckpt.best_weights is not None:
        model.set_weights(subject_ckpt.best_weights)

    # ----------------------------
    # Epoch-level evaluation on test
    # ----------------------------
    res_test = model.evaluate(ds_test, return_dict=True, verbose=0)
    epoch_test_acc = float(res_test["acc"])
    fold_epoch_test_acc.append(epoch_test_acc)

    # ----------------------------
    # Subject-level evaluation
    # ----------------------------
    val_probs, val_y = collect_subject_probs(model, val_paths, val_labels, N, E, MAX_PEAKS)
    test_probs, test_y = collect_subject_probs(model, test_paths, test_labels, N, E, MAX_PEAKS)

    best_thr = subject_ckpt.best_thr
    fold_best_thresholds.append(best_thr)

    subj_val_acc = subject_accuracy_from_probs(val_probs, val_y, threshold=best_thr)
    subj_test_acc = subject_accuracy_from_probs(test_probs, test_y, threshold=best_thr)
    subj_test_bal_acc = balanced_accuracy_score(test_y, (test_probs >= best_thr).astype(np.int32))

    try:
        subj_test_auc = roc_auc_score(test_y, test_probs)
    except Exception:
        subj_test_auc = np.nan

    fold_subject_val_acc.append(subj_val_acc)
    fold_subject_test_acc.append(subj_test_acc)
    fold_subject_test_bal_acc.append(subj_test_bal_acc)
    fold_subject_test_auc.append(subj_test_auc)

    fold_total_elapsed = time.perf_counter() - fold_start_time
    fold_total_times.append(fold_total_elapsed)

    print(f"Fold {fold} results:")
    print(f"  Epoch-level TEST acc:           {epoch_test_acc*100:.2f}%")
    print(f"  Best subject-level val thr:     {best_thr:.3f}")
    print(f"  Subject-level VAL acc:          {subj_val_acc*100:.2f}%")
    print(f"  Subject-level TEST acc:         {subj_test_acc*100:.2f}%")
    print(f"  Subject-level TEST bal. acc:    {subj_test_bal_acc*100:.2f}%")
    print(f"  Subject-level TEST AUC:         {subj_test_auc:.4f}")
    print(
        f"  Fold model.fit training time:   "
        f"{format_time(training_elapsed)} ({training_elapsed:.2f} sec)"
    )
    print(
        f"  Fold total wall-clock time:     "
        f"{format_time(fold_total_elapsed)} ({fold_total_elapsed:.2f} sec)"
    )

# ============================================================
# SUMMARY
# ============================================================
total_run_elapsed = time.perf_counter() - total_run_start_time
total_training_elapsed = float(np.sum(fold_training_times))

print(f"\n{'='*70}")
print(f"FINAL RESULTS ({K_FOLDS}-Fold, SUBJECT-WISE) - HiGMR-Net")
print(f"{'='*70}")

print("Epoch-level TEST acc per fold:", [round(a * 100, 2) for a in fold_epoch_test_acc])
print("Subject-level VAL acc per fold:", [round(a * 100, 2) for a in fold_subject_val_acc])
print("Subject-level TEST acc per fold:", [round(a * 100, 2) for a in fold_subject_test_acc])
print("Subject-level TEST bal acc per fold:", [round(a * 100, 2) for a in fold_subject_test_bal_acc])
print("Subject-level TEST AUC per fold:", [round(float(a), 4) if not np.isnan(a) else None for a in fold_subject_test_auc])
print("Best thresholds per fold:", [round(t, 3) for t in fold_best_thresholds])

print(f"\nMean epoch-TEST acc:      {np.mean(fold_epoch_test_acc)*100:.2f}% ± {np.std(fold_epoch_test_acc)*100:.2f}%")
print(f"Mean subj-VAL acc:        {np.mean(fold_subject_val_acc)*100:.2f}% ± {np.std(fold_subject_val_acc)*100:.2f}%")
print(f"Mean subj-TEST acc:       {np.mean(fold_subject_test_acc)*100:.2f}% ± {np.std(fold_subject_test_acc)*100:.2f}%")
print(f"Mean subj-TEST bal acc:   {np.mean(fold_subject_test_bal_acc)*100:.2f}% ± {np.std(fold_subject_test_bal_acc)*100:.2f}%")
print(f"Mean subj-TEST AUC:       {np.nanmean(fold_subject_test_auc):.4f} ± {np.nanstd(fold_subject_test_auc):.4f}")

print("\nTraining time per fold:")
for i, t in enumerate(fold_training_times, start=1):
    print(f"  Fold {i}: {format_time(t)} ({t:.2f} sec)")

print("\nTotal fold wall-clock time per fold:")
for i, t in enumerate(fold_total_times, start=1):
    print(f"  Fold {i}: {format_time(t)} ({t:.2f} sec)")

print(
    f"\nTotal model.fit training time across folds: "
    f"{format_time(total_training_elapsed)} ({total_training_elapsed:.2f} sec)"
)
print(
    f"Total cross-validation wall-clock time:     "
    f"{format_time(total_run_elapsed)} ({total_run_elapsed:.2f} sec)"
)

summary_path = OUT_DIR / "kfold_subjectwise_higmrnet_results.txt"
with open(summary_path, "w") as f:
    f.write(f"Epoch-level TEST acc: {fold_epoch_test_acc}\n")
    f.write("\nTraining time per fold seconds:\n")
    f.write(f"{fold_training_times}\n")
    f.write("Training time per fold formatted:\n")
    f.write(f"{[format_time(t) for t in fold_training_times]}\n")
    f.write(f"Total model.fit training time seconds: {total_training_elapsed}\n")
    f.write(f"Total model.fit training time formatted: {format_time(total_training_elapsed)}\n")

    f.write("\nTotal fold wall-clock time seconds:\n")
    f.write(f"{fold_total_times}\n")
    f.write("Total fold wall-clock time formatted:\n")
    f.write(f"{[format_time(t) for t in fold_total_times]}\n")
    f.write(f"Total cross-validation wall-clock time seconds: {total_run_elapsed}\n")
    f.write(f"Total cross-validation wall-clock time formatted: {format_time(total_run_elapsed)}\n")

print("\nSaved:", summary_path)